# v1.1 Metric Methodology

This notebook establishes the analytical definitions used by the Seattle Public
Safety Dashboard v1.1 before KPI cards, comparison tables, rankings, and
response-time metrics are implemented.

## Questions

1. How should the current analysis period be compared with the immediately
   preceding equal-length period?
2. What is the correct counting unit for crime totals?
3. How should raw and percentage changes be calculated?
4. How should crime rates per 100,000 residents be calculated?
5. How should neighborhood crime-volume rankings and rank changes work?
6. How should map coverage / unmappable-crime percentages be calculated?
7. What exactly should count as a shooting?
8. What constitutes a qualified CAD response-time observation?
9. Which call priorities should contribute to response-time KPIs?
10. How should neighborhood response-time rankings be calculated?
11. Which unified dashboard controls can legitimately affect each metric?

Production KPI logic should not be implemented until the methodology in this
notebook has been reviewed.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display


# -------------------------------------------------------------------
# Locate repository root.
# -------------------------------------------------------------------

cwd = Path.cwd().resolve()

REPO_ROOT = next(
    (
        path
        for path in [cwd, *cwd.parents]
        if (path / "dashboard").is_dir()
        and (path / "app.py").exists()
    ),
    None,
)

if REPO_ROOT is None:
    raise RuntimeError(
        "Could not locate repository root. "
        "Expected app.py and dashboard/."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository root: {REPO_ROOT}")


# -------------------------------------------------------------------
# Crime dashboard imports
# -------------------------------------------------------------------

from dashboard.crime_dashboard_data import (
    load_crime_dashboard_context,
    EVENT_ID_COLUMN as CRIME_EVENT_ID_COLUMN,
    ROW_ID_COLUMN as CRIME_REPORT_ID_COLUMN,
    TIME_COLUMN as CRIME_TIME_COLUMN,
    REPORT_TIME_COLUMN as CRIME_REPORT_TIME_COLUMN,
    CATEGORY_COLUMN as CRIME_CATEGORY_COLUMN,
    SUB_CATEGORY_COLUMN as CRIME_SUBCATEGORY_COLUMN,
    LAT_COL as CRIME_LAT_COLUMN,
    LON_COL as CRIME_LON_COLUMN,
    normalize_neighborhood_name,
)


# -------------------------------------------------------------------
# Calls dashboard imports
# -------------------------------------------------------------------

from dashboard.spd_dashboard_data import (
    load_dashboard_context as load_calls_dashboard_context,
)

from dashboard.spd_config import (
    EVENT_ID_COLUMN as CALL_EVENT_ID_COLUMN,
    ROW_ID_COLUMN as CALL_DISPATCH_ID_COLUMN,
    TIME_COLUMN as CALL_TIME_COLUMN,
    ARRIVAL_TIME_COLUMN as CALL_ARRIVAL_COLUMN,
    LAT_COL as CALL_LAT_COLUMN,
    LON_COL as CALL_LON_COLUMN,
)

from dashboard.crime_dashboard_data import (
    load_crime_dashboard_context,
    EVENT_ID_COLUMN as CRIME_EVENT_ID_COLUMN,
    ROW_ID_COLUMN as CRIME_REPORT_ID_COLUMN,
    TIME_COLUMN as CRIME_TIME_COLUMN,
    REPORT_TIME_COLUMN as CRIME_REPORT_TIME_COLUMN,
    CATEGORY_COLUMN as CRIME_CATEGORY_COLUMN,
    SUB_CATEGORY_COLUMN as CRIME_SUBCATEGORY_COLUMN,
    NEIGHBORHOOD_COLUMN as CRIME_SOURCE_NEIGHBORHOOD_COLUMN,
    LAT_COL as CRIME_LAT_COLUMN,
    LON_COL as CRIME_LON_COLUMN,
    normalize_neighborhood_name,
)

# -------------------------------------------------------------------
# Display configuration
# -------------------------------------------------------------------

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)


# -------------------------------------------------------------------
# Helpers
# -------------------------------------------------------------------

def clean_string(series: pd.Series) -> pd.Series:
    return (
        series
        .astype("string")
        .str.strip()
        .str.lower()
    )


INVALID_TEXT_VALUES = {
    "",
    "-",
    "unknown",
    "nan",
    "none",
    "<na>",
}


def valid_string_mask(series: pd.Series) -> pd.Series:
    cleaned = clean_string(series)

    return (
        cleaned.notna()
        & ~cleaned.isin(INVALID_TEXT_VALUES)
    )


# -------------------------------------------------------------------
# Load the same contexts used by the dashboards.
# -------------------------------------------------------------------

crime_context = load_crime_dashboard_context()
calls_context = load_calls_dashboard_context()

crime_snapshot = crime_context["df"].copy()
crime = crime_context["valid_time"].copy()

mappable_crime = crime_context["mappable_events"].copy()

# -------------------------------------------------------------------
# Production population snapshot
#
# Analytical denominator policy:
# - MCPP rates use calibrated ACS-based MCPP population estimates.
# - Citywide rates use the direct ACS Seattle city estimate.
# - OFM is not used as an analytical rate denominator.
# -------------------------------------------------------------------

POPULATION_SNAPSHOT_PATH = (
    REPO_ROOT
    / "data"
    / "processed"
    / "population"
    / "population_estimates.parquet"
)

if not POPULATION_SNAPSHOT_PATH.exists():
    raise FileNotFoundError(
        "Production population snapshot was not found at "
        f"{POPULATION_SNAPSHOT_PATH}. "
        "Do not fall back to the legacy neighborhood population CSV."
    )


population_snapshot = pd.read_parquet(
    POPULATION_SNAPSHOT_PATH
)


required_population_columns = {
    "geography_type",
    "geography_name",
    "population",
    "population_raw",
    "population_year",
    "source",
    "source_vintage",
    "estimation_method",
}

missing_population_columns = (
    required_population_columns
    - set(population_snapshot.columns)
)

if missing_population_columns:
    raise ValueError(
        "Population snapshot is missing required columns: "
        f"{sorted(missing_population_columns)}"
    )


population_snapshot["geography_type"] = clean_string(
    population_snapshot["geography_type"]
)

population_snapshot["geography_name"] = (
    normalize_neighborhood_name(
        population_snapshot["geography_name"]
    )
)

population_snapshot["population"] = pd.to_numeric(
    population_snapshot["population"],
    errors="coerce",
)

population_snapshot["population_raw"] = pd.to_numeric(
    population_snapshot["population_raw"],
    errors="coerce",
)

population_snapshot["population_year"] = pd.to_numeric(
    population_snapshot["population_year"],
    errors="coerce",
)


mcpp_population = (
    population_snapshot.loc[
        population_snapshot["geography_type"].eq("mcpp")
    ]
    .copy()
)

city_population_rows = (
    population_snapshot.loc[
        population_snapshot["geography_type"].eq("city")
        & population_snapshot["geography_name"].eq("seattle")
    ]
    .copy()
)


if len(city_population_rows) != 1:
    raise ValueError(
        "Expected exactly one Seattle city population row; "
        f"found {len(city_population_rows)}."
    )


if mcpp_population["geography_name"].duplicated().any():
    duplicated = (
        mcpp_population.loc[
            mcpp_population["geography_name"].duplicated(
                keep=False
            ),
            "geography_name",
        ]
        .sort_values()
        .unique()
        .tolist()
    )

    raise ValueError(
        "Duplicate MCPP population rows found: "
        f"{duplicated}"
    )


if mcpp_population["population"].isna().any():
    raise ValueError(
        "One or more MCPP population values are missing."
    )


if (mcpp_population["population"] <= 0).any():
    raise ValueError(
        "One or more MCPP population values are non-positive."
    )


city_population_row = city_population_rows.iloc[0]

CITYWIDE_POPULATION = float(
    city_population_row["population"]
)

POPULATION_YEAR = int(
    city_population_row["population_year"]
)

POPULATION_SOURCE = (
    city_population_row["source"]
)

POPULATION_SOURCE_VINTAGE = (
    city_population_row["source_vintage"]
)

POPULATION_SOURCE_CONFIRMED = True


# Keep a convenient MCPP-only DataFrame for later methodology cells.
population = (
    mcpp_population[
        [
            "geography_name",
            "population",
            "population_raw",
            "population_year",
            "source",
            "source_vintage",
            "estimation_method",
        ]
    ]
    .rename(
        columns={
            "geography_name": "mcpp_neighborhood",
        }
    )
    .copy()
)

calls = calls_context["df"].copy()
existing_response_analysis = (
    calls_context["response_analysis"].copy()
)


# -------------------------------------------------------------------
# Normalize dates.
# -------------------------------------------------------------------

crime[CRIME_TIME_COLUMN] = pd.to_datetime(
    crime[CRIME_TIME_COLUMN],
    errors="coerce",
)

crime["analysis_date"] = (
    crime[CRIME_TIME_COLUMN]
    .dt.normalize()
)

crime[CRIME_EVENT_ID_COLUMN] = clean_string(
    crime[CRIME_EVENT_ID_COLUMN]
)

crime[CRIME_REPORT_ID_COLUMN] = clean_string(
    crime[CRIME_REPORT_ID_COLUMN]
)

crime[CRIME_SUBCATEGORY_COLUMN] = clean_string(
    crime[CRIME_SUBCATEGORY_COLUMN]
)

crime["event_importance_bin"] = clean_string(
    crime["event_importance_bin"]
)


calls[CALL_TIME_COLUMN] = pd.to_datetime(
    calls[CALL_TIME_COLUMN],
    errors="coerce",
)

calls[CALL_ARRIVAL_COLUMN] = pd.to_datetime(
    calls[CALL_ARRIVAL_COLUMN],
    errors="coerce",
)

calls[CALL_EVENT_ID_COLUMN] = clean_string(
    calls[CALL_EVENT_ID_COLUMN]
)

# -------------------------------------------------------------------
# Canonical v1.1 crime types.
# -------------------------------------------------------------------

CRIME_TYPE_COLUMN = "event_importance_bin"

CANONICAL_CRIME_TYPES = [
    "crimes against persons",
    "crimes against property",
    "crimes against society / other",
]

observed_crime_types = sorted(
    crime[CRIME_TYPE_COLUMN]
    .dropna()
    .unique()
    .tolist()
)

print()
print("Observed analytical crime types:")
for value in observed_crime_types:
    print(f"  {value}")

unexpected_types = sorted(
    set(observed_crime_types)
    - set(CANONICAL_CRIME_TYPES)
)

if unexpected_types:
    print()
    print("WARNING: Non-canonical crime types remain:")
    for value in unexpected_types:
        print(f"  {value}")
else:
    print()
    print("PASS: All observed crime types are canonical v1.1 bins.")

Repository root: C:\Users\benca\code\PersonalPythonProjects\SPDCallDashboard

Observed analytical crime types:
  crimes against persons
  crimes against property
  crimes against society / other

PASS: All observed crime types are canonical v1.1 bins.


In [2]:
print("POST-CLASSIFICATION CRIME POPULATION")
print("=" * 60)

print(
    f"Unique offenses: "
    f"{crime[CRIME_EVENT_ID_COLUMN].nunique():,}"
)

print(
    f"Unique reports:  "
    f"{crime[CRIME_REPORT_ID_COLUMN].nunique():,}"
)

print(
    f"Crime date range: "
    f"{crime['analysis_date'].min().date()} "
    f"to {crime['analysis_date'].max().date()}"
)


# -------------------------------------------------------------------
# Verify that explicit NIBRS "not a crime" records are no longer
# entering the analytical population after the prior notebook's rules.
# -------------------------------------------------------------------

if "nibrs_crime_against_category" in crime.columns:
    crime_against = (
        crime["nibrs_crime_against_category"]
        .astype("string")
        .str.strip()
        .str.lower()
        .str.replace(r"[\s\-]+", "_", regex=True)
    )

    remaining_not_a_crime = (
        crime_against.eq("not_a_crime")
    )

    print(
        f"Explicit not_a_crime offenses remaining: "
        f"{crime.loc[remaining_not_a_crime, CRIME_EVENT_ID_COLUMN].nunique():,}"
    )

    if remaining_not_a_crime.any():
        print(
            "WARNING: not_a_crime records still exist in the "
            "analytical population."
        )
    else:
        print(
            "PASS: No explicit not_a_crime records remain "
            "in the analytical population."
        )


# -------------------------------------------------------------------
# Every included offense should have exactly one canonical type.
# -------------------------------------------------------------------

missing_type = ~valid_string_mask(
    crime[CRIME_TYPE_COLUMN]
)

noncanonical_type = (
    ~crime[CRIME_TYPE_COLUMN]
    .isin(CANONICAL_CRIME_TYPES)
)

print(
    f"Missing crime type: "
    f"{crime.loc[missing_type, CRIME_EVENT_ID_COLUMN].nunique():,}"
)

print(
    f"Non-canonical crime type: "
    f"{crime.loc[noncanonical_type, CRIME_EVENT_ID_COLUMN].nunique():,}"
)

POST-CLASSIFICATION CRIME POPULATION
Unique offenses: 138,894
Unique reports:  125,150
Crime date range: 2024-09-09 to 2026-09-13
Explicit not_a_crime offenses remaining: 0
PASS: No explicit not_a_crime records remain in the analytical population.
Missing crime type: 0
Non-canonical crime type: 0


## Current vs previous period methodology

The comparison period is the immediately preceding period containing the exact
same number of calendar days as the selected current period.

Both start and end dates are inclusive.

Example:

Current:
September 1 through September 7 = 7 days

Previous:
August 25 through August 31 = 7 days

There is no gap or overlap between the two periods.

Percentage change is:

(current - previous) / previous × 100

If the previous value is zero, percentage change is undefined rather than
infinite.

Stored history != selectable analysis history. The selectable domain ends at
analysis_end = latest_available_date and begins at
analysis_start = analysis_end - pd.DateOffset(years=1), with no +1 day.
Both endpoints are inclusive, matching native Plotly calendar-year semantics.
A leap-year interval can contain 367 inclusive calendar dates, so a fixed
366-day cap is incorrect.

The user selects only the current period within that domain. The previous
period is automatically derived and may precede analysis_start. Both periods
contain the same number of calendar dates, with no gap and no overlap.
Full stored history remains available for these hidden comparisons.


In [3]:
def make_equal_periods(
    current_start,
    current_end,
):
    current_start = pd.Timestamp(
        current_start
    ).normalize()

    current_end = pd.Timestamp(
        current_end
    ).normalize()

    if current_start > current_end:
        raise ValueError(
            "current_start must be <= current_end"
        )

    period_days = (
        current_end - current_start
    ).days + 1

    previous_end = (
        current_start
        - pd.Timedelta(days=1)
    )

    previous_start = (
        current_start
        - pd.Timedelta(days=period_days)
    )

    result = {
        "current_start": current_start,
        "current_end": current_end,
        "previous_start": previous_start,
        "previous_end": previous_end,
        "period_days": period_days,
    }

    # Internal correctness checks.
    previous_days = (
        previous_end - previous_start
    ).days + 1

    assert previous_days == period_days

    assert (
        previous_end
        + pd.Timedelta(days=1)
        == current_start
    )

    return result


# -------------------------------------------------------------------
# Basic unit checks.
# -------------------------------------------------------------------

one_day = make_equal_periods(
    "2026-09-06",
    "2026-09-06",
)

assert one_day["previous_start"] == pd.Timestamp(
    "2026-09-05"
)

assert one_day["previous_end"] == pd.Timestamp(
    "2026-09-05"
)


seven_days = make_equal_periods(
    "2026-08-31",
    "2026-09-06",
)

assert seven_days["period_days"] == 7

assert seven_days["previous_start"] == pd.Timestamp(
    "2026-08-24"
)

assert seven_days["previous_end"] == pd.Timestamp(
    "2026-08-30"
)

print("PASS: Equal-period helper checks passed.")

PASS: Equal-period helper checks passed.


In [4]:
# A native calendar year can contain 367 inclusive dates.
leap_end = pd.Timestamp("2024-02-29")
leap_start = leap_end - pd.DateOffset(years=1)
leap_periods = make_equal_periods(leap_start, leap_end)

assert leap_start == pd.Timestamp("2023-02-28")
assert leap_periods["period_days"] == 367
assert leap_periods["previous_start"] == pd.Timestamp("2022-02-26")
assert leap_periods["previous_end"] == pd.Timestamp("2023-02-27")
assert (leap_periods["previous_end"] - leap_periods["previous_start"]).days + 1 == 367
assert leap_periods["previous_end"] + pd.Timedelta(days=1) == leap_start
print("PASS: Native leap-year current and previous periods each contain 367 dates.")


PASS: Native leap-year current and previous periods each contain 367 dates.


In [5]:
LATEST_CRIME_DATE = (
    crime["analysis_date"]
    .dropna()
    .max()
)

EARLIEST_CRIME_DATE = (
    crime["analysis_date"]
    .dropna()
    .min()
)

CURRENT_END = LATEST_CRIME_DATE

CURRENT_START = CURRENT_END - pd.DateOffset(years=1)

periods = make_equal_periods(
    CURRENT_START,
    CURRENT_END,
)

display(
    pd.DataFrame(
        {
            "period": [
                "Current",
                "Previous",
            ],
            "start": [
                periods["current_start"],
                periods["previous_start"],
            ],
            "end": [
                periods["current_end"],
                periods["previous_end"],
            ],
            "days": [
                periods["period_days"],
                periods["period_days"],
            ],
        }
    )
)

previous_supported = (
    periods["previous_start"]
    >= EARLIEST_CRIME_DATE
)

print(
    f"Previous period fully supported: "
    f"{previous_supported}"
)

,period,start,end,days
0,Current,2025-09-13,2026-09-13,366
1,Previous,2024-09-12,2025-09-12,366


Previous period fully supported: True


In [6]:
latest_call_date = (
    calls[CALL_TIME_COLUMN]
    .dropna()
    .dt.normalize()
    .max()
)

earliest_call_date = (
    calls[CALL_TIME_COLUMN]
    .dropna()
    .dt.normalize()
    .min()
)


source_coverage = pd.DataFrame(
    {
        "source": [
            "Crime offenses",
            "CAD calls",
        ],
        "earliest_date": [
            EARLIEST_CRIME_DATE,
            earliest_call_date,
        ],
        "latest_date": [
            LATEST_CRIME_DATE,
            latest_call_date,
        ],
    }
)

source_coverage["days_available"] = (
    source_coverage["latest_date"]
    - source_coverage["earliest_date"]
).dt.days + 1

display(source_coverage)


# -------------------------------------------------------------------
# Can a full latest calendar year also have an equal previous period?
# -------------------------------------------------------------------

latest_year_start = (
    LATEST_CRIME_DATE
    - pd.DateOffset(years=1)
)

latest_year_periods = make_equal_periods(
    latest_year_start,
    LATEST_CRIME_DATE,
)

full_year_comparison_supported = (
    latest_year_periods["previous_start"]
    >= EARLIEST_CRIME_DATE
)

print(
    "Latest-year current period:"
)

print(
    latest_year_periods["current_start"].date(),
    "to",
    latest_year_periods["current_end"].date(),
)

print(
    "Required previous period begins:",
    latest_year_periods["previous_start"].date(),
)

print(
    "Full one-year comparison supported:",
    full_year_comparison_supported,
)

,source,earliest_date,latest_date,days_available
0,Crime offenses,2024-09-09,2026-09-13,735
1,CAD calls,2024-09-06,2026-09-10,735


Latest-year current period:
2025-09-13 to 2026-09-13
Required previous period begins: 2024-09-12
Full one-year comparison supported: True


In [7]:
SELECTED_CRIME_TYPES = (
    CANONICAL_CRIME_TYPES.copy()
)

SELECTED_SUBCATEGORIES = []

SELECTED_NEIGHBORHOODS = []


def filter_crime_period(
    records,
    start_date,
    end_date,
    crime_types=None,
    subcategories=None,
    neighborhoods=None,
):
    out = records.copy()

    start_date = pd.Timestamp(
        start_date
    ).normalize()

    end_date = pd.Timestamp(
        end_date
    ).normalize()

    dates = pd.to_datetime(
        out[CRIME_TIME_COLUMN],
        errors="coerce",
    ).dt.normalize()

    mask = dates.between(
        start_date,
        end_date,
    )

    if crime_types:
        selected_types = (
            clean_string(
                pd.Series(
                    crime_types,
                    dtype="string",
                )
            )
            .dropna()
            .tolist()
        )

        mask &= (
            clean_string(
                out[CRIME_TYPE_COLUMN]
            )
            .isin(selected_types)
        )

    if subcategories:
        selected_subcategories = (
            clean_string(
                pd.Series(
                    subcategories,
                    dtype="string",
                )
            )
            .dropna()
            .tolist()
        )

        mask &= (
            clean_string(
                out[CRIME_SUBCATEGORY_COLUMN]
            )
            .isin(selected_subcategories)
        )

    if neighborhoods:
        selected_neighborhoods = (
            normalize_neighborhood_name(
                pd.Series(
                    neighborhoods,
                    dtype="string",
                )
            )
            .dropna()
            .tolist()
        )

        record_neighborhoods = (
            normalize_neighborhood_name(
                out["mcpp_neighborhood"]
            )
        )

        mask &= record_neighborhoods.isin(
            selected_neighborhoods
        )

    return out.loc[mask].copy()


current_crime = filter_crime_period(
    crime,
    periods["current_start"],
    periods["current_end"],
    crime_types=SELECTED_CRIME_TYPES,
    subcategories=SELECTED_SUBCATEGORIES,
    neighborhoods=SELECTED_NEIGHBORHOODS,
)

previous_crime = filter_crime_period(
    crime,
    periods["previous_start"],
    periods["previous_end"],
    crime_types=SELECTED_CRIME_TYPES,
    subcategories=SELECTED_SUBCATEGORIES,
    neighborhoods=SELECTED_NEIGHBORHOODS,
)

print(
    f"Current offenses: "
    f"{current_crime[CRIME_EVENT_ID_COLUMN].nunique():,}"
)

print(
    f"Previous offenses: "
    f"{previous_crime[CRIME_EVENT_ID_COLUMN].nunique():,}"
)

Current offenses: 66,964
Previous offenses: 71,446


## Crime counting unit

The headline crime KPI should count unique `offense_id` values.

`report_number` represents a different concept: a single report may contain
multiple separately classified offenses.

Therefore:

**Total Offenses** = unique `offense_id`

**Reports / Incidents** = unique `report_number`

The two measures should never be used interchangeably.

Current-vs-previous crime comparisons use unique offenses.

In [8]:
def safe_pct_change(
    current,
    previous,
):
    if pd.isna(current) or pd.isna(previous):
        return np.nan

    if previous == 0:
        if current == 0:
            return 0.0

        return np.nan

    return (
        (current - previous)
        / previous
        * 100
    )


current_offenses = (
    current_crime[
        CRIME_EVENT_ID_COLUMN
    ]
    .nunique()
)

previous_offenses = (
    previous_crime[
        CRIME_EVENT_ID_COLUMN
    ]
    .nunique()
)

current_reports = (
    current_crime[
        CRIME_REPORT_ID_COLUMN
    ]
    .nunique()
)

previous_reports = (
    previous_crime[
        CRIME_REPORT_ID_COLUMN
    ]
    .nunique()
)


overall_period_comparison = pd.DataFrame(
    {
        "metric": [
            "Unique offenses",
            "Unique reports",
        ],
        "current": [
            current_offenses,
            current_reports,
        ],
        "previous": [
            previous_offenses,
            previous_reports,
        ],
    }
)

overall_period_comparison["raw_change"] = (
    overall_period_comparison["current"]
    - overall_period_comparison["previous"]
)

overall_period_comparison["pct_change"] = (
    overall_period_comparison.apply(
        lambda row: safe_pct_change(
            row["current"],
            row["previous"],
        ),
        axis=1,
    )
)

display(overall_period_comparison)

,metric,current,previous,raw_change,pct_change
0,Unique offenses,66964,71446,-4482,-6.273269
1,Unique reports,60406,64315,-3909,-6.077898


In [9]:
current_by_type = (
    current_crime
    .groupby(
        CRIME_TYPE_COLUMN
    )[CRIME_EVENT_ID_COLUMN]
    .nunique()
    .reindex(
        CANONICAL_CRIME_TYPES,
        fill_value=0,
    )
)

previous_by_type = (
    previous_crime
    .groupby(
        CRIME_TYPE_COLUMN
    )[CRIME_EVENT_ID_COLUMN]
    .nunique()
    .reindex(
        CANONICAL_CRIME_TYPES,
        fill_value=0,
    )
)


crime_type_comparison = pd.DataFrame(
    {
        "crime_type": (
            CANONICAL_CRIME_TYPES
        ),
        "current_offenses": (
            current_by_type.values
        ),
        "previous_offenses": (
            previous_by_type.values
        ),
    }
)

crime_type_comparison["raw_change"] = (
    crime_type_comparison["current_offenses"]
    - crime_type_comparison["previous_offenses"]
)

crime_type_comparison["pct_change"] = (
    crime_type_comparison.apply(
        lambda row: safe_pct_change(
            row["current_offenses"],
            row["previous_offenses"],
        ),
        axis=1,
    )
)

crime_type_comparison[
    "current_share_pct"
] = (
    100
    * crime_type_comparison[
        "current_offenses"
    ]
    / max(
        crime_type_comparison[
            "current_offenses"
        ].sum(),
        1,
    )
)

display(crime_type_comparison)

,crime_type,current_offenses,previous_offenses,raw_change,pct_change,current_share_pct
0,crimes against persons,12230,12194,36,0.295227,18.263545
1,crimes against property,44237,49598,-5361,-10.808904,66.060869
2,crimes against society / other,10497,9654,843,8.732132,15.675587


## Crime-rate methodology

For an arbitrary selected analysis period:

**crime rate per 100,000 residents**

$\
  \text{rate} =
  \frac{\text{unique offenses during selected period}}
  {\text{population denominator}}
  \times 100{,}000
$

The rate describes the selected period exactly as chosen by the user. It is **not
annualized**.

### Population denominator

Population denominators come from the production population snapshot established
by the v1.1 population methodology.

- **Citywide Seattle rate:** direct 2024 ACS 5-Year Seattle population estimate.
- **MCPP neighborhood rate:** calibrated 2024 ACS-based MCPP population estimate.
- `population_raw` is retained for QA only and is not the production denominator.
- OFM population estimates may be shown as contextual/current population
  information, but are not used as analytical rate denominators.

Current and previous periods use the same population vintage. Because the two
periods are always equal in duration, their period-specific rates can be compared
directly.

Percentage changes in rates are calculated using the same zero-denominator rules
as the other comparison metrics.

Neighborhood population coverage must reconcile to all analytical MCPP
neighborhoods before neighborhood rates are considered valid.

Very small residential denominators can produce extreme neighborhood rates.
The rate calculation itself remains mathematically valid, but rate-based
neighborhood ranking should not be exposed until a separate low-population /
nonresidential-neighborhood ranking policy is finalized.

In [10]:
population_by_neighborhood = (
    population[
        [
            "mcpp_neighborhood",
            "population",
            "population_raw",
        ]
    ]
    .copy()
)


print("POPULATION DENOMINATOR AUDIT")
print("=" * 60)

print(
    f"Population vintage: "
    f"{POPULATION_YEAR}"
)

print(
    f"Population source: "
    f"{POPULATION_SOURCE}"
)

print(
    f"Source vintage: "
    f"{POPULATION_SOURCE_VINTAGE}"
)

print()

print(
    f"MCPP population rows: "
    f"{len(population_by_neighborhood):,}"
)

print(
    f"Calibrated MCPP population total: "
    f"{population_by_neighborhood['population'].sum():,.0f}"
)

print(
    f"Raw reconstructed MCPP total: "
    f"{population_by_neighborhood['population_raw'].sum():,.0f}"
)

print(
    f"Direct Seattle population denominator: "
    f"{CITYWIDE_POPULATION:,.0f}"
)

print(
    f"Missing MCPP population values: "
    f"{population_by_neighborhood['population'].isna().sum():,}"
)

print(
    f"Zero/non-positive MCPP population values: "
    f"{(population_by_neighborhood['population'] <= 0).sum():,}"
)

# -------------------------------------------------------------------
# Validate population coverage against the canonical MCPP geography.
#
# Do NOT use crime["mcpp_neighborhood"] for this check because
# valid_time intentionally falls back to the source crime-neighborhood
# field for offenses without a spatial MCPP match.
# -------------------------------------------------------------------

boundary_neighborhoods = set(
    normalize_neighborhood_name(
        crime_context["mcpp_boundaries"][
            "mcpp_neighborhood"
        ]
    )
    .dropna()
    .loc[
        lambda s: ~s.isin(
            INVALID_TEXT_VALUES
        )
    ]
    .unique()
)

population_neighborhoods = set(
    population_by_neighborhood[
        "mcpp_neighborhood"
    ]
    .dropna()
    .unique()
)


mcpp_without_population = sorted(
    boundary_neighborhoods
    - population_neighborhoods
)

population_without_mcpp = sorted(
    population_neighborhoods
    - boundary_neighborhoods
)


print()

print(
    "Canonical MCPP neighborhoods:",
    len(boundary_neighborhoods),
)

print(
    "Population MCPP neighborhoods:",
    len(population_neighborhoods),
)

print(
    "MCPP neighborhoods without population:",
    len(mcpp_without_population),
)

print(
    "Population neighborhoods without MCPP boundary:",
    len(population_without_mcpp),
)


if mcpp_without_population:
    display(
        pd.DataFrame(
            {
                "mcpp_without_population": (
                    mcpp_without_population
                )
            }
        )
    )


if population_without_mcpp:
    display(
        pd.DataFrame(
            {
                "population_without_mcpp": (
                    population_without_mcpp
                )
            }
        )
    )


if (
    mcpp_without_population
    or population_without_mcpp
):
    raise ValueError(
        "Population snapshot does not reconcile "
        "with the canonical MCPP geography."
    )


print()
print(
    "PASS: Production population denominators "
    "cover the canonical MCPP geography."
)

POPULATION DENOMINATOR AUDIT
Population vintage: 2024
Population source: U.S. Census Bureau ACS 5-Year
Source vintage: 2024

MCPP population rows: 58
Calibrated MCPP population total: 754,194
Raw reconstructed MCPP total: 754,133
Direct Seattle population denominator: 754,195
Missing MCPP population values: 0
Zero/non-positive MCPP population values: 0

Canonical MCPP neighborhoods: 58
Population MCPP neighborhoods: 58
MCPP neighborhoods without population: 0
Population neighborhoods without MCPP boundary: 0

PASS: Production population denominators cover the canonical MCPP geography.


In [11]:
def rate_per_100k(
    offense_count,
    population_value,
):
    if (
        population_value is None
        or pd.isna(population_value)
        or population_value <= 0
    ):
        return np.nan

    return (
        offense_count
        / population_value
        * 100_000
    )


# -------------------------------------------------------------------
# Overall citywide crime-rate comparison.
#
# Rates are period-specific, not annualized.
# Current and previous periods use the same 2024 ACS denominator.
# -------------------------------------------------------------------

current_city_offenses = (
    current_crime[
        CRIME_EVENT_ID_COLUMN
    ]
    .nunique()
)

previous_city_offenses = (
    previous_crime[
        CRIME_EVENT_ID_COLUMN
    ]
    .nunique()
)


current_city_rate = rate_per_100k(
    current_city_offenses,
    CITYWIDE_POPULATION,
)

previous_city_rate = rate_per_100k(
    previous_city_offenses,
    CITYWIDE_POPULATION,
)


overall_city_rate_comparison = pd.DataFrame(
    {
        "metric": [
            "Overall crime rate per 100,000"
        ],
        "population_year": [
            POPULATION_YEAR
        ],
        "population_denominator": [
            CITYWIDE_POPULATION
        ],
        "current_offenses": [
            current_city_offenses
        ],
        "previous_offenses": [
            previous_city_offenses
        ],
        "current_rate_per_100k": [
            current_city_rate
        ],
        "previous_rate_per_100k": [
            previous_city_rate
        ],
    }
)


overall_city_rate_comparison[
    "raw_rate_change"
] = (
    overall_city_rate_comparison[
        "current_rate_per_100k"
    ]
    - overall_city_rate_comparison[
        "previous_rate_per_100k"
    ]
)


overall_city_rate_comparison[
    "pct_rate_change"
] = (
    overall_city_rate_comparison.apply(
        lambda row: safe_pct_change(
            row["current_rate_per_100k"],
            row["previous_rate_per_100k"],
        ),
        axis=1,
    )
)


display(overall_city_rate_comparison)

,metric,population_year,population_denominator,current_offenses,previous_offenses,current_rate_per_100k,previous_rate_per_100k,raw_rate_change,pct_rate_change
0,"Overall crime rate per 100,000",2024,754195.0,66964,71446,8878.870849,9473.146865,-594.276016,-6.273269


In [12]:
overall_count_pct_change = safe_pct_change(
    current_city_offenses,
    previous_city_offenses,
)

overall_rate_pct_change = (
    overall_city_rate_comparison.loc[
        0,
        "pct_rate_change",
    ]
)


print(
    "Offense-count % change:",
    overall_count_pct_change,
)

print(
    "Crime-rate % change:",
    overall_rate_pct_change,
)


assert np.isclose(
    overall_count_pct_change,
    overall_rate_pct_change,
    equal_nan=True,
)


print()
print(
    "PASS: Rate % change matches offense-count % change "
    "when the same population denominator is used."
)

Offense-count % change: -6.273269322285363
Crime-rate % change: -6.2732693222853655

PASS: Rate % change matches offense-count % change when the same population denominator is used.


### Citywide crime rate

The citywide crime rate is calculated as:

rate per 100,000 =
(unique analytical offenses during the selected period /
 direct ACS Seattle population estimate)
× 100,000

The denominator is the direct 2024 ACS 5-Year Seattle population estimate
(754,195). The sum of calibrated MCPP population estimates is not used as the
citywide denominator.

The rate applies to the user-selected analysis period and is not annualized.
All analytical offenses are retained in the citywide numerator regardless of
whether they can be assigned to an MCPP neighborhood.

Current and immediately preceding comparison periods use the same population
vintage. Therefore, for equal-length periods, percentage change in the
citywide crime rate is mathematically equal to percentage change in the
underlying unique-offense count.

In [13]:
# -------------------------------------------------------------------
# Neighborhood assignment provenance
#
# Goal:
# Separate spatially verified MCPP assignments from textual
# source-neighborhood fallbacks before defining neighborhood-rate
# numerators.
# -------------------------------------------------------------------

canonical_mcpp = set(
    normalize_neighborhood_name(
        crime_context[
            "mcpp_boundaries"
        ]["mcpp_neighborhood"]
    )
    .dropna()
    .loc[
        lambda s: ~s.isin(
            INVALID_TEXT_VALUES
        )
    ]
    .unique()
)


# -------------------------------------------------------------------
# Source neighborhood labels, reduced to offense level.
# -------------------------------------------------------------------

source_neighborhood_rows = (
    crime[
        [
            CRIME_EVENT_ID_COLUMN,
            CRIME_SOURCE_NEIGHBORHOOD_COLUMN,
        ]
    ]
    .copy()
)


source_neighborhood_rows[
    "source_neighborhood"
] = normalize_neighborhood_name(
    source_neighborhood_rows[
        CRIME_SOURCE_NEIGHBORHOOD_COLUMN
    ]
)


valid_source_mask = valid_string_mask(
    source_neighborhood_rows[
        "source_neighborhood"
    ]
)

source_neighborhood_rows.loc[
    ~valid_source_mask,
    "source_neighborhood",
] = pd.NA


source_neighborhood_rows = (
    source_neighborhood_rows[
        [
            CRIME_EVENT_ID_COLUMN,
            "source_neighborhood",
        ]
    ]
    .drop_duplicates()
)


# Number of distinct valid source-neighborhood labels per offense.
source_variant_counts = (
    source_neighborhood_rows
    .dropna(
        subset=["source_neighborhood"]
    )
    .groupby(
        CRIME_EVENT_ID_COLUMN
    )["source_neighborhood"]
    .nunique()
)


ambiguous_source_ids = set(
    source_variant_counts.loc[
        source_variant_counts > 1
    ].index
)


print(
    "Offenses with >1 distinct source neighborhood:",
    len(ambiguous_source_ids),
)


# Only assign a source fallback when an offense has exactly one
# distinct valid source-neighborhood label.
source_single = (
    source_neighborhood_rows.loc[
        ~source_neighborhood_rows[
            CRIME_EVENT_ID_COLUMN
        ].isin(ambiguous_source_ids)
    ]
    .dropna(
        subset=["source_neighborhood"]
    )
    .drop_duplicates(
        subset=[CRIME_EVENT_ID_COLUMN]
    )
)


# -------------------------------------------------------------------
# Spatial MCPP lookup.
# -------------------------------------------------------------------

spatial_lookup = (
    crime_context[
        "event_mcpp_lookup"
    ][
        [
            CRIME_EVENT_ID_COLUMN,
            "mcpp_neighborhood",
        ]
    ]
    .copy()
)


spatial_lookup[
    "spatial_mcpp_neighborhood"
] = normalize_neighborhood_name(
    spatial_lookup[
        "mcpp_neighborhood"
    ]
)


spatial_lookup = (
    spatial_lookup[
        [
            CRIME_EVENT_ID_COLUMN,
            "spatial_mcpp_neighborhood",
        ]
    ]
    .drop_duplicates(
        subset=[CRIME_EVENT_ID_COLUMN]
    )
)


# -------------------------------------------------------------------
# One row per analytical offense.
# -------------------------------------------------------------------

neighborhood_assignment = pd.DataFrame(
    {
        CRIME_EVENT_ID_COLUMN: (
            crime[
                CRIME_EVENT_ID_COLUMN
            ]
            .dropna()
            .unique()
        )
    }
)


neighborhood_assignment = (
    neighborhood_assignment
    .merge(
        source_single[
            [
                CRIME_EVENT_ID_COLUMN,
                "source_neighborhood",
            ]
        ],
        on=CRIME_EVENT_ID_COLUMN,
        how="left",
    )
    .merge(
        spatial_lookup,
        on=CRIME_EVENT_ID_COLUMN,
        how="left",
    )
)


neighborhood_assignment[
    "source_is_canonical_mcpp"
] = (
    neighborhood_assignment[
        "source_neighborhood"
    ]
    .isin(canonical_mcpp)
)


neighborhood_assignment[
    "spatial_is_canonical_mcpp"
] = (
    neighborhood_assignment[
        "spatial_mcpp_neighborhood"
    ]
    .isin(canonical_mcpp)
)


neighborhood_assignment[
    "source_is_ambiguous"
] = (
    neighborhood_assignment[
        CRIME_EVENT_ID_COLUMN
    ]
    .isin(ambiguous_source_ids)
)


# -------------------------------------------------------------------
# Assignment provenance.
# -------------------------------------------------------------------

neighborhood_assignment[
    "assignment_provenance"
] = np.select(
    [
        neighborhood_assignment[
            "spatial_is_canonical_mcpp"
        ],

        (
            ~neighborhood_assignment[
                "spatial_is_canonical_mcpp"
            ]
            & neighborhood_assignment[
                "source_is_canonical_mcpp"
            ]
        ),

        (
            ~neighborhood_assignment[
                "spatial_is_canonical_mcpp"
            ]
            & neighborhood_assignment[
                "source_neighborhood"
            ].notna()
        ),

        neighborhood_assignment[
            "source_is_ambiguous"
        ],
    ],
    [
        "spatial_mcpp",
        "canonical_source_fallback",
        "noncanonical_source_only",
        "ambiguous_source_neighborhood",
    ],
    default="no_usable_neighborhood",
)


assignment_summary = (
    neighborhood_assignment[
        "assignment_provenance"
    ]
    .value_counts()
    .rename_axis(
        "assignment_provenance"
    )
    .reset_index(
        name="offenses"
    )
)


assignment_summary[
    "share_pct"
] = (
    assignment_summary["offenses"]
    / len(neighborhood_assignment)
    * 100
)


print()
print(
    "Unique analytical offenses:",
    f"{len(neighborhood_assignment):,}",
)

display(assignment_summary)

Offenses with >1 distinct source neighborhood: 0

Unique analytical offenses: 138,894


,assignment_provenance,offenses,share_pct
0,spatial_mcpp,117823,84.829438
1,canonical_source_fallback,19896,14.324593
2,no_usable_neighborhood,1173,0.844529
3,noncanonical_source_only,2,0.001440


In [14]:
# -------------------------------------------------------------------
# Agreement between spatial MCPP assignment and source neighborhood.
# -------------------------------------------------------------------

both_assignments = (
    neighborhood_assignment.loc[
        neighborhood_assignment[
            "spatial_is_canonical_mcpp"
        ]
        & neighborhood_assignment[
            "source_is_canonical_mcpp"
        ]
    ]
    .copy()
)


both_assignments[
    "assignments_agree"
] = (
    both_assignments[
        "spatial_mcpp_neighborhood"
    ]
    == both_assignments[
        "source_neighborhood"
    ]
)


agreement_count = (
    both_assignments[
        "assignments_agree"
    ].sum()
)

comparison_count = len(
    both_assignments
)

agreement_pct = (
    agreement_count
    / comparison_count
    * 100
    if comparison_count
    else np.nan
)


print(
    "Offenses with both spatial and canonical "
    "source assignments:",
    f"{comparison_count:,}",
)

print(
    "Assignments agreeing:",
    f"{agreement_count:,}",
)

print(
    "Agreement rate:",
    f"{agreement_pct:.2f}%",
)


mismatches = (
    both_assignments.loc[
        ~both_assignments[
            "assignments_agree"
        ]
    ]
)


print(
    "Assignments disagreeing:",
    f"{len(mismatches):,}",
)


mismatch_pairs = (
    mismatches
    .groupby(
        [
            "source_neighborhood",
            "spatial_mcpp_neighborhood",
        ],
        as_index=False,
    )
    .agg(
        offenses=(
            CRIME_EVENT_ID_COLUMN,
            "nunique",
        )
    )
    .sort_values(
        "offenses",
        ascending=False,
    )
)


display(
    mismatch_pairs.head(30)
)

Offenses with both spatial and canonical source assignments: 116,901
Assignments agreeing: 110,546
Agreement rate: 94.56%
Assignments disagreeing: 6,355


,source_neighborhood,spatial_mcpp_neighborhood,offenses
154,north beacon hill,mount baker,378
19,bitterlake,northgate,302
162,northgate,bitterlake,254
16,belltown,slu/cascade,246
33,central area/squire park,miller park,233
26,capitol hill,first hill,181
184,queen anne,slu/cascade,166
194,roosevelt/ravenna,university,159
177,pioneer square,downtown commercial,133
206,slu/cascade,belltown,118


In [15]:
noncanonical_source_labels = (
    neighborhood_assignment.loc[
        neighborhood_assignment[
            "assignment_provenance"
        ].eq(
            "noncanonical_source_only"
        ),
        "source_neighborhood",
    ]
    .value_counts()
    .rename_axis(
        "source_neighborhood"
    )
    .reset_index(
        name="offenses"
    )
)


display(
    noncanonical_source_labels.head(50)
)

,source_neighborhood,offenses
0,ooj,2


In [16]:
# -------------------------------------------------------------------
# Final neighborhood assignment for rate methodology.
#
# Priority:
# 1. Spatially verified canonical MCPP.
# 2. Canonical source-neighborhood fallback only when spatial
#    assignment is unavailable.
# 3. Otherwise unassigned.
# -------------------------------------------------------------------

neighborhood_assignment[
    "rate_mcpp_neighborhood"
] = pd.NA


spatial_mask = (
    neighborhood_assignment[
        "spatial_is_canonical_mcpp"
    ]
)

source_fallback_mask = (
    ~spatial_mask
    & neighborhood_assignment[
        "source_is_canonical_mcpp"
    ]
)


neighborhood_assignment.loc[
    spatial_mask,
    "rate_mcpp_neighborhood",
] = neighborhood_assignment.loc[
    spatial_mask,
    "spatial_mcpp_neighborhood",
]


neighborhood_assignment.loc[
    source_fallback_mask,
    "rate_mcpp_neighborhood",
] = neighborhood_assignment.loc[
    source_fallback_mask,
    "source_neighborhood",
]


neighborhood_assignment[
    "rate_assignment_method"
] = np.select(
    [
        spatial_mask,
        source_fallback_mask,
    ],
    [
        "spatial",
        "canonical_source_fallback",
    ],
    default="unassigned",
)


rate_assignment_summary = (
    neighborhood_assignment[
        "rate_assignment_method"
    ]
    .value_counts()
    .rename_axis(
        "rate_assignment_method"
    )
    .reset_index(
        name="offenses"
    )
)


rate_assignment_summary[
    "share_pct"
] = (
    rate_assignment_summary[
        "offenses"
    ]
    / len(neighborhood_assignment)
    * 100
)


assigned_offenses = (
    neighborhood_assignment[
        "rate_mcpp_neighborhood"
    ]
    .notna()
    .sum()
)

assignment_coverage_pct = (
    assigned_offenses
    / len(neighborhood_assignment)
    * 100
)


display(rate_assignment_summary)

print()
print(
    "Neighborhood-rate numerator coverage:",
    f"{assigned_offenses:,} / "
    f"{len(neighborhood_assignment):,} "
    f"({assignment_coverage_pct:.2f}%)",
)

,rate_assignment_method,offenses,share_pct
0,spatial,117823,84.829438
1,canonical_source_fallback,19896,14.324593
2,unassigned,1175,0.845969



Neighborhood-rate numerator coverage: 137,719 / 138,894 (99.15%)


In [17]:
def build_neighborhood_rate_period(
    period_crime,
    assignment,
    population_frame,
):
    period_offenses = (
        period_crime[
            [CRIME_EVENT_ID_COLUMN]
        ]
        .dropna()
        .drop_duplicates()
        .merge(
            assignment[
                [
                    CRIME_EVENT_ID_COLUMN,
                    "rate_mcpp_neighborhood",
                    "rate_assignment_method",
                ]
            ],
            on=CRIME_EVENT_ID_COLUMN,
            how="left",
        )
    )


    assigned = (
        period_offenses.loc[
            period_offenses[
                "rate_mcpp_neighborhood"
            ].notna()
        ]
        .copy()
    )


    counts = (
        assigned
        .groupby(
            "rate_mcpp_neighborhood",
            as_index=False,
        )
        .agg(
            offenses=(
                CRIME_EVENT_ID_COLUMN,
                "nunique",
            )
        )
    )


    result = (
        population_frame[
            [
                "mcpp_neighborhood",
                "population",
            ]
        ]
        .merge(
            counts,
            left_on="mcpp_neighborhood",
            right_on="rate_mcpp_neighborhood",
            how="left",
        )
    )


    result["offenses"] = (
        result["offenses"]
        .fillna(0)
        .astype(int)
    )


    result["rate_per_100k"] = (
        result.apply(
            lambda row: rate_per_100k(
                row["offenses"],
                row["population"],
            ),
            axis=1,
        )
    )


    total_offenses = len(period_offenses)
    assigned_offense_count = len(assigned)

    coverage_pct = (
        assigned_offense_count
        / total_offenses
        * 100
        if total_offenses
        else np.nan
    )


    return (
        result,
        {
            "total_offenses": total_offenses,
            "assigned_offenses": assigned_offense_count,
            "coverage_pct": coverage_pct,
        },
    )


current_neighborhood_rates, current_rate_coverage = (
    build_neighborhood_rate_period(
        current_crime,
        neighborhood_assignment,
        population_by_neighborhood,
    )
)


previous_neighborhood_rates, previous_rate_coverage = (
    build_neighborhood_rate_period(
        previous_crime,
        neighborhood_assignment,
        population_by_neighborhood,
    )
)


print(
    "Current-period neighborhood assignment coverage:",
    f"{current_rate_coverage['coverage_pct']:.2f}%",
)

print(
    "Previous-period neighborhood assignment coverage:",
    f"{previous_rate_coverage['coverage_pct']:.2f}%",
)

Current-period neighborhood assignment coverage: 98.82%
Previous-period neighborhood assignment coverage: 99.47%


In [18]:
neighborhood_rate_comparison = (
    current_neighborhood_rates[
        [
            "mcpp_neighborhood",
            "population",
            "offenses",
            "rate_per_100k",
        ]
    ]
    .rename(
        columns={
            "offenses": "current_offenses",
            "rate_per_100k": "current_rate_per_100k",
        }
    )
    .merge(
        previous_neighborhood_rates[
            [
                "mcpp_neighborhood",
                "offenses",
                "rate_per_100k",
            ]
        ]
        .rename(
            columns={
                "offenses": "previous_offenses",
                "rate_per_100k": "previous_rate_per_100k",
            }
        ),
        on="mcpp_neighborhood",
        how="outer",
    )
)


neighborhood_rate_comparison[
    "raw_rate_change"
] = (
    neighborhood_rate_comparison[
        "current_rate_per_100k"
    ]
    - neighborhood_rate_comparison[
        "previous_rate_per_100k"
    ]
)


neighborhood_rate_comparison[
    "pct_rate_change"
] = (
    neighborhood_rate_comparison.apply(
        lambda row: safe_pct_change(
            row["current_rate_per_100k"],
            row["previous_rate_per_100k"],
        ),
        axis=1,
    )
)


display(
    neighborhood_rate_comparison.sort_values(
        "current_rate_per_100k",
        ascending=False,
    )
)

,mcpp_neighborhood,population,current_offenses,current_rate_per_100k,previous_offenses,previous_rate_per_100k,raw_rate_change,pct_rate_change
13,commercial harbor island,3.0,33,1.100000e+06,24,800000.000000,300000.000000,37.500000
12,commercial duwamish,24.0,111,4.625000e+05,92,383333.333333,79166.666667,20.652174
52,sodo,615.0,1010,1.642276e+05,1303,211869.918699,-47642.276423,-22.486569
44,pioneer square,2100.0,1302,6.200000e+04,1163,55380.952381,6619.047619,11.951849
21,georgetown,1341.0,729,5.436242e+04,892,66517.524236,-12155.108128,-18.273543
9,chinatown/international district,5850.0,2344,4.006838e+04,2292,39179.487179,888.888889,2.268761
14,downtown commercial,9401.0,3465,3.685778e+04,3842,40867.992767,-4010.211680,-9.812598
4,belltown,11501.0,2087,1.814625e+04,1988,17285.453439,860.794714,4.979879
7,capitol hill,32036.0,5356,1.671869e+04,5690,17761.268573,-1042.577101,-5.869947
18,first hill,18344.0,2858,1.558003e+04,2798,15252.943742,327.082425,2.144389


In [19]:
def build_neighborhood_assignment_diagnostics(
    period_crime,
    assignment,
):
    period_ids = (
        period_crime[
            [CRIME_EVENT_ID_COLUMN]
        ]
        .dropna()
        .drop_duplicates()
    )

    period_assignments = (
        period_ids
        .merge(
            assignment[
                [
                    CRIME_EVENT_ID_COLUMN,
                    "rate_mcpp_neighborhood",
                    "rate_assignment_method",
                ]
            ],
            on=CRIME_EVENT_ID_COLUMN,
            how="left",
        )
    )

    assigned = (
        period_assignments.loc[
            period_assignments[
                "rate_mcpp_neighborhood"
            ].notna()
        ]
        .copy()
    )

    diagnostics = (
        assigned
        .groupby(
            "rate_mcpp_neighborhood",
            as_index=False,
        )
        .agg(
            assigned_offenses=(
                CRIME_EVENT_ID_COLUMN,
                "nunique",
            ),
            spatial_offenses=(
                "rate_assignment_method",
                lambda s: (s == "spatial").sum(),
            ),
            fallback_offenses=(
                "rate_assignment_method",
                lambda s: (
                    s == "canonical_source_fallback"
                ).sum(),
            ),
        )
    )

    diagnostics[
        "spatial_share_pct"
    ] = (
        diagnostics["spatial_offenses"]
        / diagnostics["assigned_offenses"]
        * 100
    )

    diagnostics[
        "fallback_share_pct"
    ] = (
        diagnostics["fallback_offenses"]
        / diagnostics["assigned_offenses"]
        * 100
    )

    return diagnostics


current_assignment_diagnostics = (
    build_neighborhood_assignment_diagnostics(
        current_crime,
        neighborhood_assignment,
    )
)


neighborhood_rate_diagnostics = (
    neighborhood_rate_comparison
    .merge(
        current_assignment_diagnostics,
        left_on="mcpp_neighborhood",
        right_on="rate_mcpp_neighborhood",
        how="left",
    )
    .drop(
        columns=["rate_mcpp_neighborhood"],
        errors="ignore",
    )
)


# How much does ONE additional offense change the rate?
neighborhood_rate_diagnostics[
    "rate_increment_per_offense"
] = (
    100_000
    / neighborhood_rate_diagnostics[
        "population"
    ]
)


display(
    neighborhood_rate_diagnostics[
        [
            "mcpp_neighborhood",
            "population",
            "current_offenses",
            "current_rate_per_100k",
            "rate_increment_per_offense",
            "spatial_share_pct",
            "fallback_share_pct",
        ]
    ]
    .sort_values(
        "current_rate_per_100k",
        ascending=False,
    )
)

,mcpp_neighborhood,population,current_offenses,current_rate_per_100k,rate_increment_per_offense,spatial_share_pct,fallback_share_pct
13,commercial harbor island,3.0,33,1.100000e+06,33333.333333,84.848485,15.151515
12,commercial duwamish,24.0,111,4.625000e+05,4166.666667,86.486486,13.513514
52,sodo,615.0,1010,1.642276e+05,162.601626,91.287129,8.712871
44,pioneer square,2100.0,1302,6.200000e+04,47.619048,77.956989,22.043011
21,georgetown,1341.0,729,5.436242e+04,74.571216,90.123457,9.876543
9,chinatown/international district,5850.0,2344,4.006838e+04,17.094017,82.380546,17.619454
14,downtown commercial,9401.0,3465,3.685778e+04,10.637166,82.222222,17.777778
4,belltown,11501.0,2087,1.814625e+04,8.694896,82.702444,17.297556
7,capitol hill,32036.0,5356,1.671869e+04,3.121488,82.991038,17.008962
18,first hill,18344.0,2858,1.558003e+04,5.451374,81.420574,18.579426


In [20]:
candidate_population_thresholds = [
    1_000,
    2_000,
    4_000,
    5_000,
    7_500,
    10_000,
]


threshold_rows = []


for threshold in candidate_population_thresholds:
    eligible = (
        neighborhood_rate_diagnostics[
            "population"
        ] >= threshold
    )

    excluded_names = (
        neighborhood_rate_diagnostics.loc[
            ~eligible,
            "mcpp_neighborhood",
        ]
        .sort_values()
        .tolist()
    )

    threshold_rows.append(
        {
            "minimum_population": threshold,
            "max_rate_increment_per_offense": (
                100_000 / threshold
            ),
            "eligible_neighborhoods": int(
                eligible.sum()
            ),
            "excluded_neighborhoods": int(
                (~eligible).sum()
            ),
            "excluded_names": ", ".join(
                excluded_names
            ),
        }
    )


population_threshold_sensitivity = (
    pd.DataFrame(threshold_rows)
)


display(
    population_threshold_sensitivity
)

,minimum_population,max_rate_increment_per_offense,eligible_neighborhoods,excluded_neighborhoods,excluded_names
0,1000,100.000000,55,3,"commercial duwamish, commercial harbor island, sodo"
1,2000,50.000000,50,8,"commercial duwamish, commercial harbor island, eastlake - east, genesee, georgetown, pigeon point, sodo, south delridge"
2,4000,25.000000,49,9,"commercial duwamish, commercial harbor island, eastlake - east, genesee, georgetown, pigeon point, pioneer square, s..."
3,5000,20.000000,46,12,"commercial duwamish, commercial harbor island, eastlake - east, genesee, georgetown, hillman city, pigeon point, pio..."
4,7500,13.333333,34,24,"chinatown/international district, claremont/rainier vista, columbia city, commercial duwamish, commercial harbor isl..."
5,10000,10.000000,29,29,"alki, chinatown/international district, claremont/rainier vista, columbia city, commercial duwamish, commercial harb..."


In [21]:
def show_rate_ranking_at_population_threshold(
    diagnostics,
    minimum_population,
    top_n=15,
):
    ranked = (
        diagnostics.loc[
            diagnostics[
                "population"
            ] >= minimum_population
        ]
        .sort_values(
            "current_rate_per_100k",
            ascending=False,
        )
        .copy()
    )

    print(
        f"Minimum population: "
        f"{minimum_population:,}"
    )

    print(
        "Maximum rate increment from one offense:",
        f"{100_000 / minimum_population:,.2f} "
        "per 100,000",
    )

    print(
        "Eligible neighborhoods:",
        len(ranked),
    )

    display(
        ranked[
            [
                "mcpp_neighborhood",
                "population",
                "current_offenses",
                "current_rate_per_100k",
                "rate_increment_per_offense",
                "spatial_share_pct",
                "fallback_share_pct",
            ]
        ]
        .head(top_n)
    )


show_rate_ranking_at_population_threshold(
    neighborhood_rate_diagnostics,
    4_000,
)

show_rate_ranking_at_population_threshold(
    neighborhood_rate_diagnostics,
    5_000,
)

Minimum population: 4,000
Maximum rate increment from one offense: 25.00 per 100,000
Eligible neighborhoods: 49


,mcpp_neighborhood,population,current_offenses,current_rate_per_100k,rate_increment_per_offense,spatial_share_pct,fallback_share_pct
9,chinatown/international district,5850.0,2344,40068.376068,17.094017,82.380546,17.619454
14,downtown commercial,9401.0,3465,36857.781087,10.637166,82.222222,17.777778
4,belltown,11501.0,2087,18146.248152,8.694896,82.702444,17.297556
7,capitol hill,32036.0,5356,16718.691472,3.121488,82.991038,17.008962
18,first hill,18344.0,2858,15580.026167,5.451374,81.420574,18.579426
36,mount baker,8723.0,1283,14708.242577,11.463946,85.035074,14.964926
26,judkins park/north beacon hill,5372.0,774,14408.041698,18.615041,83.074935,16.925065
46,rainier beach,5777.0,819,14176.908430,17.310023,73.504274,26.495726
51,slu/cascade,25464.0,3507,13772.384543,3.927113,86.626747,13.373253
3,ballard south,26048.0,2705,10384.674447,3.839066,87.208872,12.791128


Minimum population: 5,000
Maximum rate increment from one offense: 20.00 per 100,000
Eligible neighborhoods: 46


,mcpp_neighborhood,population,current_offenses,current_rate_per_100k,rate_increment_per_offense,spatial_share_pct,fallback_share_pct
9,chinatown/international district,5850.0,2344,40068.376068,17.094017,82.380546,17.619454
14,downtown commercial,9401.0,3465,36857.781087,10.637166,82.222222,17.777778
4,belltown,11501.0,2087,18146.248152,8.694896,82.702444,17.297556
7,capitol hill,32036.0,5356,16718.691472,3.121488,82.991038,17.008962
18,first hill,18344.0,2858,15580.026167,5.451374,81.420574,18.579426
36,mount baker,8723.0,1283,14708.242577,11.463946,85.035074,14.964926
26,judkins park/north beacon hill,5372.0,774,14408.041698,18.615041,83.074935,16.925065
46,rainier beach,5777.0,819,14176.908430,17.310023,73.504274,26.495726
51,slu/cascade,25464.0,3507,13772.384543,3.927113,86.626747,13.373253
3,ballard south,26048.0,2705,10384.674447,3.839066,87.208872,12.791128


In [22]:
MIN_RATE_RANK_POPULATION = 5_000


neighborhood_rate_diagnostics[
    "rate_rank_eligible"
] = (
    neighborhood_rate_diagnostics[
        "population"
    ] >= MIN_RATE_RANK_POPULATION
)


neighborhood_rate_diagnostics[
    "rate_rank_exclusion_reason"
] = np.where(
    neighborhood_rate_diagnostics[
        "rate_rank_eligible"
    ],
    pd.NA,
    "population below rate-ranking threshold",
)


neighborhood_rate_diagnostics[
    "current_rate_rank"
] = pd.NA


eligible_mask = (
    neighborhood_rate_diagnostics[
        "rate_rank_eligible"
    ]
)


neighborhood_rate_diagnostics.loc[
    eligible_mask,
    "current_rate_rank",
] = (
    neighborhood_rate_diagnostics.loc[
        eligible_mask,
        "current_rate_per_100k",
    ]
    .rank(
        method="min",
        ascending=False,
    )
)


display(
    neighborhood_rate_diagnostics[
        [
            "mcpp_neighborhood",
            "population",
            "current_offenses",
            "current_rate_per_100k",
            "rate_rank_eligible",
            "current_rate_rank",
            "rate_rank_exclusion_reason",
        ]
    ]
    .sort_values(
        [
            "rate_rank_eligible",
            "current_rate_rank",
        ],
        ascending=[
            False,
            True,
        ],
    )
)

,mcpp_neighborhood,population,current_offenses,current_rate_per_100k,rate_rank_eligible,current_rate_rank,rate_rank_exclusion_reason
9,chinatown/international district,5850.0,2344,4.006838e+04,True,1.0,NaN
14,downtown commercial,9401.0,3465,3.685778e+04,True,2.0,NaN
4,belltown,11501.0,2087,1.814625e+04,True,3.0,NaN
7,capitol hill,32036.0,5356,1.671869e+04,True,4.0,NaN
18,first hill,18344.0,2858,1.558003e+04,True,5.0,NaN
36,mount baker,8723.0,1283,1.470824e+04,True,6.0,NaN
26,judkins park/north beacon hill,5372.0,774,1.440804e+04,True,7.0,NaN
46,rainier beach,5777.0,819,1.417691e+04,True,8.0,NaN
51,slu/cascade,25464.0,3507,1.377238e+04,True,9.0,NaN
3,ballard south,26048.0,2705,1.038467e+04,True,10.0,NaN


In [23]:
MIN_RATE_CONTEXT_POPULATION = 5_000


neighborhood_rate_diagnostics[
    "small_population_rate_warning"
] = (
    neighborhood_rate_diagnostics[
        "population"
    ] < MIN_RATE_CONTEXT_POPULATION
)


neighborhood_rate_diagnostics[
    "rate_context_note"
] = np.where(
    neighborhood_rate_diagnostics[
        "small_population_rate_warning"
    ],
    (
        "Small residential population; "
        "per-capita rate may be highly sensitive "
        "to individual offenses."
    ),
    pd.NA,
)

In [24]:
def build_neighborhood_volume_period(
    period_crime,
    assignment,
):
    period_offenses = (
        period_crime[
            [CRIME_EVENT_ID_COLUMN]
        ]
        .dropna()
        .drop_duplicates()
        .merge(
            assignment[
                [
                    CRIME_EVENT_ID_COLUMN,
                    "rate_mcpp_neighborhood",
                ]
            ],
            on=CRIME_EVENT_ID_COLUMN,
            how="left",
        )
    )

    assigned = (
        period_offenses.loc[
            period_offenses[
                "rate_mcpp_neighborhood"
            ].notna()
        ]
        .copy()
    )

    volume = (
        assigned
        .groupby(
            "rate_mcpp_neighborhood",
            as_index=False,
        )
        .agg(
            offenses=(
                CRIME_EVENT_ID_COLUMN,
                "nunique",
            )
        )
        .rename(
            columns={
                "rate_mcpp_neighborhood":
                    "mcpp_neighborhood",
            }
        )
    )

    volume["rank"] = (
        volume["offenses"]
        .rank(
            method="min",
            ascending=False,
        )
    )

    return volume


current_neighborhood_volume = (
    build_neighborhood_volume_period(
        current_crime,
        neighborhood_assignment,
    )
)

previous_neighborhood_volume = (
    build_neighborhood_volume_period(
        previous_crime,
        neighborhood_assignment,
    )
)

In [25]:
neighborhood_volume_comparison = (
    current_neighborhood_volume
    .rename(
        columns={
            "offenses": "current_offenses",
            "rank": "current_rank",
        }
    )
    .merge(
        previous_neighborhood_volume.rename(
            columns={
                "offenses": "previous_offenses",
                "rank": "previous_rank",
            }
        ),
        on="mcpp_neighborhood",
        how="outer",
    )
)


neighborhood_volume_comparison[
    [
        "current_offenses",
        "previous_offenses",
    ]
] = (
    neighborhood_volume_comparison[
        [
            "current_offenses",
            "previous_offenses",
        ]
    ]
    .fillna(0)
    .astype(int)
)


neighborhood_volume_comparison[
    "raw_change"
] = (
    neighborhood_volume_comparison[
        "current_offenses"
    ]
    - neighborhood_volume_comparison[
        "previous_offenses"
    ]
)


neighborhood_volume_comparison[
    "pct_change"
] = (
    neighborhood_volume_comparison.apply(
        lambda row: safe_pct_change(
            row["current_offenses"],
            row["previous_offenses"],
        ),
        axis=1,
    )
)


# Positive = neighborhood moved toward rank 1.
neighborhood_volume_comparison[
    "rank_change"
] = (
    neighborhood_volume_comparison[
        "previous_rank"
    ]
    - neighborhood_volume_comparison[
        "current_rank"
    ]
)


display(
    neighborhood_volume_comparison.sort_values(
        "current_rank"
    )
)

,mcpp_neighborhood,current_offenses,current_rank,previous_offenses,previous_rank,raw_change,pct_change,rank_change
7,capitol hill,5356,1.0,5690,1.0,-334,-5.869947,0.0
45,queen anne,4010,2.0,3684,4.0,326,8.849077,2.0
51,slu/cascade,3507,3.0,3766,3.0,-259,-6.877323,0.0
14,downtown commercial,3465,4.0,3842,2.0,-377,-9.812598,-2.0
41,northgate,3174,5.0,3565,5.0,-391,-10.967742,0.0
18,first hill,2858,6.0,2798,7.0,60,2.144389,1.0
56,university,2776,7.0,2973,6.0,-197,-6.626303,-1.0
3,ballard south,2705,8.0,2668,8.0,37,1.386807,0.0
9,chinatown/international district,2344,9.0,2292,10.0,52,2.268761,1.0
48,roosevelt/ravenna,2312,10.0,2610,9.0,-298,-11.417625,-1.0


In [26]:
neighborhood_methodology_table = (
    neighborhood_volume_comparison
    .merge(
        neighborhood_rate_diagnostics[
            [
                "mcpp_neighborhood",
                "population",
                "current_rate_per_100k",
                "previous_rate_per_100k",
                "raw_rate_change",
                "pct_rate_change",
                "rate_increment_per_offense",
                "spatial_share_pct",
                "fallback_share_pct",
                "small_population_rate_warning",
                "rate_context_note",
            ]
        ],
        on="mcpp_neighborhood",
        how="left",
    )
)


display(
    neighborhood_methodology_table
    .sort_values(
        "current_rank"
    )
    .head(20)
)

,mcpp_neighborhood,current_offenses,current_rank,previous_offenses,previous_rank,raw_change,pct_change,rank_change,population,current_rate_per_100k,previous_rate_per_100k,raw_rate_change,pct_rate_change,rate_increment_per_offense,spatial_share_pct,fallback_share_pct,small_population_rate_warning,rate_context_note
7,capitol hill,5356,1.0,5690,1.0,-334,-5.869947,0.0,32036.0,16718.691472,17761.268573,-1042.577101,-5.869947,3.121488,82.991038,17.008962,False,NaN
45,queen anne,4010,2.0,3684,4.0,326,8.849077,2.0,48474.0,8272.475966,7599.950489,672.525478,8.849077,2.062962,87.880299,12.119701,False,NaN
51,slu/cascade,3507,3.0,3766,3.0,-259,-6.877323,0.0,25464.0,13772.384543,14789.506755,-1017.122212,-6.877323,3.927113,86.626747,13.373253,False,NaN
14,downtown commercial,3465,4.0,3842,2.0,-377,-9.812598,-2.0,9401.0,36857.781087,40867.992767,-4010.211680,-9.812598,10.637166,82.222222,17.777778,False,NaN
41,northgate,3174,5.0,3565,5.0,-391,-10.967742,0.0,38239.0,8300.426266,9322.942546,-1022.516279,-10.967742,2.615131,83.931947,16.068053,False,NaN
18,first hill,2858,6.0,2798,7.0,60,2.144389,1.0,18344.0,15580.026167,15252.943742,327.082425,2.144389,5.451374,81.420574,18.579426,False,NaN
56,university,2776,7.0,2973,6.0,-197,-6.626303,-1.0,32535.0,8532.349777,9137.851544,-605.501767,-6.626303,3.073613,87.968300,12.031700,False,NaN
3,ballard south,2705,8.0,2668,8.0,37,1.386807,0.0,26048.0,10384.674447,10242.628993,142.045455,1.386807,3.839066,87.208872,12.791128,False,NaN
9,chinatown/international district,2344,9.0,2292,10.0,52,2.268761,1.0,5850.0,40068.376068,39179.487179,888.888889,2.268761,17.094017,82.380546,17.619454,False,NaN
48,roosevelt/ravenna,2312,10.0,2610,9.0,-298,-11.417625,-1.0,32130.0,7195.767196,8123.249300,-927.482104,-11.417625,3.112356,89.619377,10.380623,False,NaN


## Neighborhood-level crime methodology

Neighborhood-level analysis uses Seattle Police Department Micro Community
Policing Plan (MCPP) neighborhoods as the canonical geography.

The neighborhood methodology has three distinct components:

1. assigning analytical offenses to MCPP neighborhoods,
2. calculating neighborhood offense counts and population-normalized rates,
3. ranking neighborhoods for dashboard comparison.

### Neighborhood assignment

Spatial assignment is treated as the authoritative neighborhood assignment.

For each unique analytical offense:

1. If the offense has a valid spatial match to a canonical MCPP polygon, the
   spatial MCPP assignment is used.
2. If no spatial assignment is available, the source crime-dataset neighborhood
   may be used only when its normalized value exactly matches one of the canonical
   MCPP neighborhood names.
3. If neither condition is met, the offense remains unassigned for
   neighborhood-level analysis.

The source neighborhood never overrides an available spatial MCPP assignment.

Offenses that cannot be assigned to an MCPP remain included in citywide crime
counts and citywide crime-rate calculations. They are excluded only from
neighborhood-specific numerators.

Neighborhood-assignment coverage should therefore be retained as a QA metric
alongside neighborhood analyses.

### Neighborhood offense volume

Neighborhood offense volume is the count of unique analytical offenses assigned
to each MCPP neighborhood during the selected analysis period.

The current period is compared with the immediately preceding equal-length
period.

For each neighborhood, the methodology calculates:

- current unique-offense count,
- previous unique-offense count,
- raw change,
- percentage change,
- current volume rank,
- previous volume rank,
- rank change.

A positive rank change means that the neighborhood moved toward rank 1.

Neighborhood **volume**, rather than population-normalized crime rate, is the
primary basis for ordered neighborhood rankings in the dashboard.

### Neighborhood crime rate

Neighborhood crime rates are calculated as:

$\
\text{crime rate per 100,000} =
\frac{\text{unique assigned offenses during selected period}}
{\text{estimated neighborhood population}}
\times 100{,}000
$

The population denominator is the calibrated MCPP population estimate produced
by the v1.1 population methodology.

The rate describes the selected analysis period exactly as chosen by the user
and is **not annualized**.

Current and previous periods use the same population vintage.

### Interpretation of neighborhood rates

Neighborhood crime rates are retained as contextual metrics rather than used as
the primary neighborhood ranking measure.

Residential population is not always an adequate measure of the population
exposed within an MCPP neighborhood. This is especially important for
commercial, industrial, institutional, entertainment, and high-visitor areas.

Very small residential populations can also make per-capita rates highly
sensitive to individual offenses. A mathematically valid rate can therefore be
misleading when interpreted as a direct measure of individual risk.

For that reason:

- valid neighborhood rates are still calculated,
- raw offense counts remain available,
- denominator-sensitivity diagnostics are retained for QA,
- small-population neighborhoods may receive a warning or ranking restriction,
- neighborhood crime rates should not be interpreted as direct estimates of an
  individual's probability of victimization.

The exact warning/suppression policy for small-population or strongly
nonresidential MCPP neighborhoods remains provisional and should be reviewed
before production deployment.

### Current methodological status

The current research implementation therefore uses:

- spatial MCPP assignment as authoritative,
- canonical source-neighborhood fallback only when spatial assignment is
  unavailable,
- unique offenses as the neighborhood counting unit,
- calibrated ACS-derived MCPP estimates as population denominators,
- offense volume as the primary neighborhood ranking metric,
- per-capita crime rate as a contextual metric.

These choices remain subject to review before the v1.1 methodology is considered
production-final.

## Neighborhood volume rankings

Neighborhood ranking is based on unique offenses, not raw rows and not reports.

Offenses without a valid analytical neighborhood remain in citywide totals but
cannot participate in neighborhood rankings.

Current rank:
rank neighborhoods by current-period unique offense count, descending.

Previous rank:
rank neighborhoods by previous-period unique offense count, descending.

Rank change:
previous rank - current rank

A positive rank change means the neighborhood moved closer to rank #1
(higher crime-volume ranking).

A negative rank change means it moved farther from rank #1.

In [27]:
def prepare_rankable_neighborhoods(
    records,
):
    out = records.copy()

    out["analysis_neighborhood"] = (
        normalize_neighborhood_name(
            out["mcpp_neighborhood"]
        )
    )

    valid_neighborhood = (
        out[
            "analysis_neighborhood"
        ].notna()
        & ~out[
            "analysis_neighborhood"
        ].isin(
            INVALID_TEXT_VALUES
        )
    )

    return (
        out.loc[
            valid_neighborhood
        ].copy()
    )


current_rankable = (
    prepare_rankable_neighborhoods(
        current_crime
    )
)

previous_rankable = (
    prepare_rankable_neighborhoods(
        previous_crime
    )
)


current_neighborhood_counts = (
    current_rankable
    .groupby(
        "analysis_neighborhood"
    )[CRIME_EVENT_ID_COLUMN]
    .nunique()
    .rename("current_offenses")
)

previous_neighborhood_counts = (
    previous_rankable
    .groupby(
        "analysis_neighborhood"
    )[CRIME_EVENT_ID_COLUMN]
    .nunique()
    .rename("previous_offenses")
)


neighborhood_volume = (
    pd.concat(
        [
            current_neighborhood_counts,
            previous_neighborhood_counts,
        ],
        axis=1,
    )
    .fillna(0)
    .reset_index()
)

neighborhood_volume[
    "current_offenses"
] = (
    neighborhood_volume[
        "current_offenses"
    ].astype(int)
)

neighborhood_volume[
    "previous_offenses"
] = (
    neighborhood_volume[
        "previous_offenses"
    ].astype(int)
)


neighborhood_volume["current_rank"] = (
    neighborhood_volume[
        "current_offenses"
    ]
    .rank(
        method="min",
        ascending=False,
    )
)

neighborhood_volume["previous_rank"] = (
    neighborhood_volume[
        "previous_offenses"
    ]
    .rank(
        method="min",
        ascending=False,
    )
)

neighborhood_volume["rank_change"] = (
    neighborhood_volume["previous_rank"]
    - neighborhood_volume["current_rank"]
)

neighborhood_volume["raw_change"] = (
    neighborhood_volume["current_offenses"]
    - neighborhood_volume["previous_offenses"]
)

neighborhood_volume["pct_change"] = (
    neighborhood_volume.apply(
        lambda row: safe_pct_change(
            row["current_offenses"],
            row["previous_offenses"],
        ),
        axis=1,
    )
)


top_neighborhoods = (
    neighborhood_volume
    .sort_values(
        [
            "current_rank",
            "analysis_neighborhood",
        ]
    )
    .head(10)
)

display(top_neighborhoods)


current_unrankable = (
    current_crime.loc[
        ~current_crime.index.isin(
            current_rankable.index
        ),
        CRIME_EVENT_ID_COLUMN,
    ]
    .nunique()
)

print(
    f"Current-period offenses excluded from "
    f"neighborhood rankings only: "
    f"{current_unrankable:,}"
)

,analysis_neighborhood,current_offenses,previous_offenses,current_rank,previous_rank,rank_change,raw_change,pct_change
7,capitol hill,5356,5690,1.0,1.0,0.0,-334,-5.869947
46,queen anne,4010,3684,2.0,4.0,2.0,326,8.849077
52,slu/cascade,3507,3766,3.0,3.0,0.0,-259,-6.877323
14,downtown commercial,3465,3842,4.0,2.0,-2.0,-377,-9.812598
41,northgate,3174,3565,5.0,5.0,0.0,-391,-10.967742
18,first hill,2858,2798,6.0,7.0,1.0,60,2.144389
57,university,2776,2973,7.0,6.0,-1.0,-197,-6.626303
3,ballard south,2705,2668,8.0,8.0,0.0,37,1.386807
9,chinatown/international district,2344,2292,9.0,10.0,1.0,52,2.268761
49,roosevelt/ravenna,2312,2610,10.0,9.0,-1.0,-298,-11.417625


Current-period offenses excluded from neighborhood rankings only: 792


## Map coverage

The map must not imply that all analytically counted offenses are visible as
points.

For the current selected analytical population:

unmappable percentage =
1 - (unique mappable offense IDs / total unique offense IDs)

This calculation must happen **after** the active crime filters are applied.

Unmappable offenses remain in citywide totals, time-series values, KPIs, and
period comparisons.

Only geographic visualization requires mappability.

In [ ]:
mappable_crime_ids = set(
    clean_string(
        mappable_crime[
            CRIME_EVENT_ID_COLUMN
        ]
    )
    .dropna()
    .tolist()
)


def calculate_map_coverage(
    selected_records,
):
    selected_ids = set(
        clean_string(
            selected_records[
                CRIME_EVENT_ID_COLUMN
            ]
        )
        .dropna()
        .tolist()
    )

    total = len(selected_ids)

    if total == 0:
        return {
            "total_offenses": 0,
            "mappable_offenses": 0,
            "unmappable_offenses": 0,
            "mappable_pct": np.nan,
            "unmappable_pct": np.nan,
        }

    mappable = len(
        selected_ids
        & mappable_crime_ids
    )

    unmappable = (
        total - mappable
    )

    return {
        "total_offenses": total,
        "mappable_offenses": mappable,
        "unmappable_offenses": unmappable,
        "mappable_pct": (
            100 * mappable / total
        ),
        "unmappable_pct": (
            100 * unmappable / total
        ),
    }


current_map_coverage = (
    calculate_map_coverage(
        current_crime
    )
)

previous_map_coverage = (
    calculate_map_coverage(
        previous_crime
    )
)

display(
    pd.DataFrame(
        [
            {
                "period": "Current",
                **current_map_coverage,
            },
            {
                "period": "Previous",
                **previous_map_coverage,
            },
        ]
    )
)


if not pd.isna(
    current_map_coverage[
        "unmappable_pct"
    ]
):
    MAP_ANNOTATION_TEXT = (
        f"{current_map_coverage['unmappable_pct']:.1f}% "
        "of offenses in this selection cannot be mapped. "
        "Citywide totals include these offenses."
    )

    print()
    print("Candidate map annotation:")
    print(MAP_ANNOTATION_TEXT)

,period,total_offenses,mappable_offenses,unmappable_offenses,mappable_pct,unmappable_pct
0,Current,66986,56102,10884,83.751829,16.248171
1,Previous,71498,61899,9599,86.574450,13.425550



Candidate map annotation:
16.2% of offenses in this selection cannot be mapped. Citywide totals include these offenses.


## Shooting KPI methodology

Before counting "shootings", inspect the actual values in
`shooting_type_group`.

We also need to determine the counting unit.

Because multiple offenses may exist on one report, a shooting KPI could differ
substantially depending on whether it counts:

- unique offenses, or
- unique reports / incidents.

No shooting KPI should be implemented until both the qualifying field values
and the counting unit have been reviewed.

In [ ]:
SHOOTING_COLUMN = (
    "shooting_type_group"
)

if SHOOTING_COLUMN not in crime.columns:
    raise ValueError(
        f"Missing required field: "
        f"{SHOOTING_COLUMN}"
    )


shooting_audit = crime.copy()

shooting_audit[
    "shooting_value"
] = (
    clean_string(
        shooting_audit[
            SHOOTING_COLUMN
        ]
    )
    .fillna("<missing>")
)

shooting_value_summary = (
    shooting_audit
    .groupby(
        "shooting_value",
        dropna=False,
    )
    .agg(
        rows=(
            CRIME_EVENT_ID_COLUMN,
            "size",
        ),
        unique_offenses=(
            CRIME_EVENT_ID_COLUMN,
            "nunique",
        ),
        unique_reports=(
            CRIME_REPORT_ID_COLUMN,
            "nunique",
        ),
    )
    .reset_index()
    .sort_values(
        "unique_offenses",
        ascending=False,
    )
)

shooting_value_summary[
    "offenses_per_report"
] = (
    shooting_value_summary[
        "unique_offenses"
    ]
    / shooting_value_summary[
        "unique_reports"
    ].replace(
        0,
        np.nan,
    )
)

display(shooting_value_summary)


shooting_crosswalk = (
    shooting_audit
    .groupby(
        [
            "shooting_value",
            CRIME_TYPE_COLUMN,
            CRIME_SUBCATEGORY_COLUMN,
        ],
        dropna=False,
    )
    .agg(
        unique_offenses=(
            CRIME_EVENT_ID_COLUMN,
            "nunique",
        ),
        unique_reports=(
            CRIME_REPORT_ID_COLUMN,
            "nunique",
        ),
    )
    .reset_index()
    .sort_values(
        "unique_offenses",
        ascending=False,
    )
)

display(shooting_crosswalk)

,shooting_value,rows,unique_offenses,unique_reports,offenses_per_report
0,-,137648,137648,124275,1.107608
3,shots fired (eyewitness/casings/property damage),1169,1169,859,1.360885
2,shooting (non-fatal injury),266,266,196,1.357143
1,shooting (fatal injury),63,63,43,1.465116


,shooting_value,event_importance_bin,offense_sub_category,unique_offenses,unique_reports
11,-,crimes against property,larceny-theft,46754,46569
9,-,crimes against property,burglary,15462,15462
13,-,crimes against property,"property offenses (includes stolen, destruction)",13444,13430
1,-,crimes against persons,assault offenses,13244,13137
12,-,crimes against property,motor vehicle theft,11354,11354
10,-,crimes against property,extortion/fraud/forgery/bribery (includes bad checks),6712,6376
0,-,crimes against persons,aggravated assault,5999,5999
14,-,crimes against society / other,all other,5169,5115
20,-,crimes against society / other,narcotic violations (includes drug equip.),3802,3389
25,-,crimes against society / other,trespass,3082,3081


In [ ]:
# -------------------------------------------------------------------
# Fill these only after reviewing Cell 19.
#
# Example only:
#
# SHOOTING_VALUE_DECISIONS = {
#     "some confirmed shooting value": True,
#     "some confirmed non-shooting value": False,
# }
# -------------------------------------------------------------------

SHOOTING_VALUE_DECISIONS = {
}


# Does a missing shooting_type_group mean "not a shooting"?
#
# Set to True or False only after confirming the field semantics.

MISSING_SHOOTING_MEANS_NO = None


# Final count unit should be one of:
#
# "offense"
# "report"

SHOOTING_COUNT_UNIT = None


observed_shooting_values = sorted(
    set(
        shooting_audit[
            "shooting_value"
        ].unique()
    )
    - {"<missing>"}
)

unresolved_shooting_values = [
    value
    for value in observed_shooting_values
    if value
    not in SHOOTING_VALUE_DECISIONS
]

print(
    "Unresolved shooting values:",
    unresolved_shooting_values,
)

print(
    "Missing-value semantics resolved:",
    MISSING_SHOOTING_MEANS_NO
    is not None,
)

print(
    "Shooting count unit:",
    SHOOTING_COUNT_UNIT,
)

Unresolved shooting values: ['-', 'shooting (fatal injury)', 'shooting (non-fatal injury)', 'shots fired (eyewitness/casings/property damage)']
Missing-value semantics resolved: False
Shooting count unit: None


## Qualified CAD response time

Response time is an event-level CAD metric.

Candidate event definition:

queued time = earliest queue timestamp for a CAD event

first arrival time = earliest recorded arrival timestamp for that CAD event

response time =
first arrival time - queued time

The current production pipeline accepts response times from 0 through 24 hours.

This notebook first reproduces that existing implementation, then evaluates:

- missing arrival times
- negative response times
- extreme response times
- priority composition
- alternative maximum-response thresholds
- the effect of limiting the KPI to dispatch-response priorities

Coordinates are **not** required for response-time inclusion.

In [28]:
response_source = calls[
    calls[
        CALL_EVENT_ID_COLUMN
    ].notna()
].copy()

response_source = (
    response_source
    .sort_values(
        CALL_TIME_COLUMN
    )
)


response_agg = {
    "queued_time": (
        CALL_TIME_COLUMN,
        "min",
    ),
    "first_arrival_time": (
        CALL_ARRIVAL_COLUMN,
        "min",
    ),
    "priority": (
        "priority",
        "first",
    ),
    "dispatch_neighborhood": (
        "dispatch_neighborhood",
        "first",
    ),
    "event_group": (
        "event_group",
        "first",
    ),
}

if (
    "event_importance_bin"
    in response_source.columns
):
    response_agg[
        "event_importance_bin"
    ] = (
        "event_importance_bin",
        "first",
    )


response_events = (
    response_source
    .groupby(
        CALL_EVENT_ID_COLUMN,
        as_index=False,
    )
    .agg(
        **response_agg
    )
)


response_events[
    "response_time_minutes"
] = (
    response_events[
        "first_arrival_time"
    ]
    - response_events[
        "queued_time"
    ]
).dt.total_seconds() / 60


response_events[
    "queued_date"
] = (
    pd.to_datetime(
        response_events[
            "queued_time"
        ],
        errors="coerce",
    )
    .dt.normalize()
)


response_events[
    "priority"
] = pd.to_numeric(
    response_events[
        "priority"
    ],
    errors="coerce",
)


response_events[
    "dispatch_neighborhood"
] = clean_string(
    response_events[
        "dispatch_neighborhood"
    ]
)


CURRENT_PRODUCTION_MAX_MINUTES = (
    24 * 60
)


response_events[
    "current_production_qualified"
] = (
    response_events[
        "response_time_minutes"
    ].notna()
    & response_events[
        "response_time_minutes"
    ].between(
        0,
        CURRENT_PRODUCTION_MAX_MINUTES,
    )
)


# -------------------------------------------------------------------
# Verify that our reconstruction agrees with the current production
# response-analysis population.
# -------------------------------------------------------------------

existing_response_ids = set(
    clean_string(
        existing_response_analysis[
            CALL_EVENT_ID_COLUMN
        ]
    )
    .dropna()
    .tolist()
)

reconstructed_qualified_ids = set(
    response_events.loc[
        response_events[
            "current_production_qualified"
        ],
        CALL_EVENT_ID_COLUMN,
    ]
    .dropna()
    .tolist()
)


print(
    f"Existing response-analysis events: "
    f"{len(existing_response_ids):,}"
)

print(
    f"Reconstructed current-rule events: "
    f"{len(reconstructed_qualified_ids):,}"
)

print(
    "Exact ID match:",
    existing_response_ids
    == reconstructed_qualified_ids,
)

Existing response-analysis events: 609,519
Reconstructed current-rule events: 609,519
Exact ID match: True


In [29]:
total_cad_events = len(
    response_events
)

has_queue = (
    response_events[
        "queued_time"
    ].notna()
)

has_arrival = (
    response_events[
        "first_arrival_time"
    ].notna()
)

has_calculated_response = (
    response_events[
        "response_time_minutes"
    ].notna()
)

nonnegative_response = (
    has_calculated_response
    & (
        response_events[
            "response_time_minutes"
        ]
        >= 0
    )
)

within_24_hours = (
    nonnegative_response
    & (
        response_events[
            "response_time_minutes"
        ]
        <= CURRENT_PRODUCTION_MAX_MINUTES
    )
)


response_funnel = pd.DataFrame(
    {
        "stage": [
            "Unique CAD events",
            "Has queued time",
            "Has arrival time",
            "Response can be calculated",
            "Response >= 0 minutes",
            "Response <= 24 hours",
        ],
        "events": [
            total_cad_events,
            int(has_queue.sum()),
            int(has_arrival.sum()),
            int(
                has_calculated_response.sum()
            ),
            int(
                nonnegative_response.sum()
            ),
            int(
                within_24_hours.sum()
            ),
        ],
    }
)

response_funnel[
    "share_of_all_events_pct"
] = (
    100
    * response_funnel["events"]
    / total_cad_events
)

display(response_funnel)

,stage,events,share_of_all_events_pct
0,Unique CAD events,681306,100.000000
1,Has queued time,681306,100.000000
2,Has arrival time,610444,89.599094
3,Response can be calculated,610444,89.599094
4,Response >= 0 minutes,610422,89.595864
5,Response <= 24 hours,609519,89.463325


In [30]:
PRIORITY_LABELS = {
    1: (
        "Incidents posing an imminent "
        "threat to life"
    ),
    2: (
        "Urgent (non-life threatening)"
    ),
    3: (
        "Non-emergency (Routine)"
    ),
    4: (
        "Administrative calls, cold "
        "incidents (Low-Level)"
    ),
    5: (
        "Alternative/Telephone reporting"
    ),
    7: (
        "Officer-Initiated Activity"
    ),
    9: (
        "Lowest Urgency / Information only"
    ),
}


priority_rows = []

for priority, group in (
    response_events
    .groupby(
        "priority",
        dropna=False,
    )
):
    qualified = group[
        group[
            "response_time_minutes"
        ].between(
            0,
            CURRENT_PRODUCTION_MAX_MINUTES,
        )
    ]

    negative_count = int(
        (
            group[
                "response_time_minutes"
            ]
            < 0
        ).sum()
    )

    over_24h_count = int(
        (
            group[
                "response_time_minutes"
            ]
            > CURRENT_PRODUCTION_MAX_MINUTES
        ).sum()
    )

    priority_key = (
        int(priority)
        if pd.notna(priority)
        else None
    )

    priority_rows.append(
        {
            "priority": priority,
            "label": (
                PRIORITY_LABELS.get(
                    priority_key,
                    "Unknown / missing",
                )
            ),
            "cad_events": len(group),
            "with_arrival": int(
                group[
                    "first_arrival_time"
                ].notna().sum()
            ),
            "qualified_24h": (
                len(qualified)
            ),
            "negative_response": (
                negative_count
            ),
            "over_24h": (
                over_24h_count
            ),
            "median_minutes": (
                qualified[
                    "response_time_minutes"
                ].median()
            ),
            "p90_minutes": (
                qualified[
                    "response_time_minutes"
                ].quantile(0.90)
            ),
            "p95_minutes": (
                qualified[
                    "response_time_minutes"
                ].quantile(0.95)
            ),
            "max_minutes": (
                qualified[
                    "response_time_minutes"
                ].max()
            ),
        }
    )


priority_response_summary = (
    pd.DataFrame(
        priority_rows
    )
    .sort_values(
        "priority",
        na_position="last",
    )
)

display(priority_response_summary)

,priority,label,cad_events,with_arrival,qualified_24h,negative_response,over_24h,median_minutes,p90_minutes,p95_minutes,max_minutes
0,1,Incidents posing an imminent threat to life,66287,61286,61279,7,0,6.900000,16.850000,22.233333,1333.533333
1,2,Urgent (non-life threatening),217495,201173,201125,7,41,24.866667,176.916667,268.683333,1376.483333
2,3,Non-emergency (Routine),173969,157804,157222,5,577,35.416667,355.933333,536.948333,1439.950000
3,4,"Administrative calls, cold incidents (Low-Level)",63739,31315,31053,2,260,48.550000,437.643333,627.086667,1439.483333
4,5,Alternative/Telephone reporting,47467,46534,46509,0,25,32.633333,356.990000,467.750000,1433.750000
5,7,Officer-Initiated Activity,100232,100222,100221,1,0,0.000000,0.000000,0.016667,306.200000
6,9,Lowest Urgency / Information only,12117,12110,12110,0,0,0.000000,0.000000,0.016667,481.700000


In [32]:
CANDIDATE_QUALIFIED_PRIORITIES = [
    1,
    2,
    3,
]


candidate_priority_events = (
    response_events[
        response_events[
            "priority"
        ].isin(
            CANDIDATE_QUALIFIED_PRIORITIES
        )
        & response_events[
            "response_time_minutes"
        ].notna()
        & (
            response_events[
                "response_time_minutes"
            ]
            >= 0
        )
    ]
    .copy()
)


print(
    f"Nonnegative response observations "
    f"for priorities 1-3: "
    f"{len(candidate_priority_events):,}"
)

Nonnegative response observations for priorities 1-3: 420,244


In [33]:
RESPONSE_CAPS_MINUTES = [
    60,
    120,
    180,
    240,
    480,
    1440,
]


base_candidate_count = len(
    candidate_priority_events
)

cap_rows = []

for cap in RESPONSE_CAPS_MINUTES:
    retained = (
        candidate_priority_events[
            candidate_priority_events[
                "response_time_minutes"
            ]
            <= cap
        ]
    )

    cap_rows.append(
        {
            "max_response_minutes": cap,
            "qualified_events": len(
                retained
            ),
            "retained_pct": (
                100
                * len(retained)
                / base_candidate_count
                if base_candidate_count
                else np.nan
            ),
            "median_minutes": (
                retained[
                    "response_time_minutes"
                ].median()
            ),
            "p90_minutes": (
                retained[
                    "response_time_minutes"
                ].quantile(0.90)
            ),
            "p95_minutes": (
                retained[
                    "response_time_minutes"
                ].quantile(0.95)
            ),
            "max_observed_minutes": (
                retained[
                    "response_time_minutes"
                ].max()
            ),
        }
    )


response_cap_sensitivity = (
    pd.DataFrame(
        cap_rows
    )
)

display(response_cap_sensitivity)

,max_response_minutes,qualified_events,retained_pct,median_minutes,p90_minutes,p95_minutes,max_observed_minutes
0,60,291312,69.319729,8.850000,39.450000,48.750000,60.00
1,120,340297,80.976052,11.766667,73.816667,93.750000,120.00
2,180,366096,87.115105,13.633333,102.533333,134.150000,180.00
3,240,381528,90.787257,14.950000,125.538333,169.550000,240.00
4,480,407404,96.944632,17.466667,181.766667,270.030833,480.00
5,1440,419626,99.852943,18.850000,222.533333,359.929167,1439.95


In [ ]:
# -------------------------------------------------------------------
# Candidate v1.1 methodology.
#
# Revisit MAX_QUALIFIED_RESPONSE_MINUTES after reviewing Cell 26.
# -------------------------------------------------------------------

QUALIFIED_RESPONSE_PRIORITIES = [
    1,
    2,
    3,
]

MAX_QUALIFIED_RESPONSE_MINUTES = (
    1440
)

MIN_NEIGHBORHOOD_RESPONSE_EVENTS = (
    30
)


def qualified_response_events(
    response_data,
    start_date,
    end_date,
    priorities=QUALIFIED_RESPONSE_PRIORITIES,
    max_minutes=MAX_QUALIFIED_RESPONSE_MINUTES,
):
    start_date = pd.Timestamp(
        start_date
    ).normalize()

    end_date = pd.Timestamp(
        end_date
    ).normalize()

    dates = pd.to_datetime(
        response_data[
            "queued_time"
        ],
        errors="coerce",
    ).dt.normalize()

    mask = (
        dates.between(
            start_date,
            end_date,
        )
        & response_data[
            "priority"
        ].isin(priorities)
        & response_data[
            "response_time_minutes"
        ].notna()
        & response_data[
            "response_time_minutes"
        ].between(
            0,
            max_minutes,
        )
    )

    return (
        response_data.loc[
            mask
        ].copy()
    )


current_response = (
    qualified_response_events(
        response_events,
        periods["current_start"],
        periods["current_end"],
    )
)

previous_response = (
    qualified_response_events(
        response_events,
        periods["previous_start"],
        periods["previous_end"],
    )
)


print(
    f"Current qualified CAD events: "
    f"{len(current_response):,}"
)

print(
    f"Previous qualified CAD events: "
    f"{len(previous_response):,}"
)

Current qualified CAD events: 209,590
Previous qualified CAD events: 207,509


In [ ]:
def summarize_response_period(
    response_data,
):
    if response_data.empty:
        return {
            "qualified_cad_events": 0,
            "median_response_minutes": np.nan,
            "mean_response_minutes": np.nan,
            "p90_response_minutes": np.nan,
            "p95_response_minutes": np.nan,
        }

    response_minutes = (
        response_data[
            "response_time_minutes"
        ]
    )

    return {
        "qualified_cad_events": (
            len(response_data)
        ),
        "median_response_minutes": (
            response_minutes.median()
        ),
        "mean_response_minutes": (
            response_minutes.mean()
        ),
        "p90_response_minutes": (
            response_minutes.quantile(
                0.90
            )
        ),
        "p95_response_minutes": (
            response_minutes.quantile(
                0.95
            )
        ),
    }


current_response_summary = (
    summarize_response_period(
        current_response
    )
)

previous_response_summary = (
    summarize_response_period(
        previous_response
    )
)


response_metric_rows = []

for metric in (
    current_response_summary.keys()
):
    current_value = (
        current_response_summary[
            metric
        ]
    )

    previous_value = (
        previous_response_summary[
            metric
        ]
    )

    response_metric_rows.append(
        {
            "metric": metric,
            "current": current_value,
            "previous": previous_value,
            "raw_change": (
                current_value
                - previous_value
                if (
                    pd.notna(current_value)
                    and pd.notna(
                        previous_value
                    )
                )
                else np.nan
            ),
            "pct_change": (
                safe_pct_change(
                    current_value,
                    previous_value,
                )
            ),
        }
    )


response_period_comparison = (
    pd.DataFrame(
        response_metric_rows
    )
)

display(response_period_comparison)

,metric,current,previous,raw_change,pct_change
0,qualified_cad_events,209590.000000,207509.000000,2081.000000,1.002848
1,median_response_minutes,17.850000,19.983333,-2.133333,-10.675563
2,mean_response_minutes,72.998244,82.777649,-9.779406,-11.814065
3,p90_response_minutes,207.216667,236.350000,-29.133333,-12.326352
4,p95_response_minutes,338.975833,379.093333,-40.117500,-10.582486


## Neighborhood response-time ranking

Neighborhood response-time rankings use qualified CAD events only.

They do not require usable coordinates.

A valid dispatch neighborhood is required because the metric is inherently
geographic.

Neighborhoods with too few qualified events should not be ranked because a
median based on a tiny sample is unstable.

Candidate threshold:

30 qualified CAD events during the selected period.

Worst-response rank #1 means the highest median response time.

For rank change:

previous worst-response rank - current worst-response rank

A positive value means the neighborhood moved upward toward the worse end of
the ranking.

In [ ]:
def prepare_response_neighborhood_stats(
    response_data,
    min_events=(
        MIN_NEIGHBORHOOD_RESPONSE_EVENTS
    ),
):
    out = response_data.copy()

    out[
        "analysis_neighborhood"
    ] = normalize_neighborhood_name(
        out[
            "dispatch_neighborhood"
        ]
    )

    valid_neighborhood = (
        out[
            "analysis_neighborhood"
        ].notna()
        & ~out[
            "analysis_neighborhood"
        ].isin(
            INVALID_TEXT_VALUES
        )
    )

    out = out.loc[
        valid_neighborhood
    ].copy()

    summary = (
        out
        .groupby(
            "analysis_neighborhood",
            as_index=False,
        )
        .agg(
            qualified_events=(
                CALL_EVENT_ID_COLUMN,
                "nunique",
            ),
            median_response_minutes=(
                "response_time_minutes",
                "median",
            ),
            mean_response_minutes=(
                "response_time_minutes",
                "mean",
            ),
        )
    )

    summary = summary[
        summary[
            "qualified_events"
        ]
        >= min_events
    ].copy()

    summary["worst_response_rank"] = (
        summary[
            "median_response_minutes"
        ]
        .rank(
            method="min",
            ascending=False,
        )
    )

    return summary


current_response_neighborhoods = (
    prepare_response_neighborhood_stats(
        current_response
    )
)

previous_response_neighborhoods = (
    prepare_response_neighborhood_stats(
        previous_response
    )
)


current_response_neighborhoods = (
    current_response_neighborhoods
    .rename(
        columns={
            "qualified_events": (
                "current_qualified_events"
            ),
            "median_response_minutes": (
                "current_median_minutes"
            ),
            "mean_response_minutes": (
                "current_mean_minutes"
            ),
            "worst_response_rank": (
                "current_worst_rank"
            ),
        }
    )
)

previous_response_neighborhoods = (
    previous_response_neighborhoods
    .rename(
        columns={
            "qualified_events": (
                "previous_qualified_events"
            ),
            "median_response_minutes": (
                "previous_median_minutes"
            ),
            "mean_response_minutes": (
                "previous_mean_minutes"
            ),
            "worst_response_rank": (
                "previous_worst_rank"
            ),
        }
    )
)


response_neighborhood_comparison = (
    current_response_neighborhoods
    .merge(
        previous_response_neighborhoods,
        on="analysis_neighborhood",
        how="outer",
    )
)


response_neighborhood_comparison[
    "median_change_minutes"
] = (
    response_neighborhood_comparison[
        "current_median_minutes"
    ]
    - response_neighborhood_comparison[
        "previous_median_minutes"
    ]
)

response_neighborhood_comparison[
    "median_pct_change"
] = (
    response_neighborhood_comparison
    .apply(
        lambda row: safe_pct_change(
            row[
                "current_median_minutes"
            ],
            row[
                "previous_median_minutes"
            ],
        ),
        axis=1,
    )
)

response_neighborhood_comparison[
    "rank_change_toward_worse"
] = (
    response_neighborhood_comparison[
        "previous_worst_rank"
    ]
    - response_neighborhood_comparison[
        "current_worst_rank"
    ]
)


worst_current_response = (
    response_neighborhood_comparison
    .sort_values(
        "current_worst_rank",
        na_position="last",
    )
    .head(10)
)

display(worst_current_response)

,analysis_neighborhood,current_qualified_events,current_median_minutes,current_mean_minutes,current_worst_rank,previous_qualified_events,previous_median_minutes,previous_mean_minutes,previous_worst_rank,median_change_minutes,median_pct_change,rank_change_toward_worse
16,eastlake - west,698,39.033333,132.143983,1.0,698,58.358333,147.441428,1.0,-19.325000,-33.114380,0.0
31,magnolia,2281,32.916667,109.809046,2.0,2196,31.058333,104.983250,18.0,1.858333,5.983365,16.0
57,wallingford,2560,32.150000,100.687982,3.0,2403,51.450000,116.329664,2.0,-19.300000,-37.512148,-1.0
30,madrona/leschi,1861,31.483333,86.159117,4.0,1877,32.200000,80.561668,16.0,-0.716667,-2.225673,12.0
19,fremont,2791,31.183333,103.903929,5.0,2449,46.966667,124.713740,3.0,-15.783333,-33.605394,-2.0
3,ballard south,6949,29.800000,99.776599,6.0,6943,42.983333,111.880654,6.0,-13.183333,-30.670803,0.0
33,miller park,1363,29.633333,80.180117,7.0,1285,37.366667,98.068573,11.0,-7.733333,-20.695807,4.0
2,ballard north,3098,29.341667,100.940784,8.0,2862,44.375000,124.565246,4.0,-15.033333,-33.877934,-4.0
48,roosevelt/ravenna,5577,28.833333,100.846232,9.0,5191,44.150000,126.173014,5.0,-15.316667,-34.692337,-4.0
29,madison park,429,28.400000,74.766006,10.0,485,37.900000,97.057491,10.0,-9.500000,-25.065963,0.0


In [ ]:
crime_neighborhood_set = set(
    normalize_neighborhood_name(
        crime[
            "mcpp_neighborhood"
        ]
    )
    .dropna()
    .loc[
        lambda s: ~s.isin(
            INVALID_TEXT_VALUES
        )
    ]
    .unique()
)


call_neighborhood_set = set(
    normalize_neighborhood_name(
        calls[
            "dispatch_neighborhood"
        ]
    )
    .dropna()
    .loc[
        lambda s: ~s.isin(
            INVALID_TEXT_VALUES
        )
    ]
    .unique()
)


shared_neighborhoods = (
    crime_neighborhood_set
    & call_neighborhood_set
)

crime_only_neighborhoods = sorted(
    crime_neighborhood_set
    - call_neighborhood_set
)

calls_only_neighborhoods = sorted(
    call_neighborhood_set
    - crime_neighborhood_set
)


print(
    f"Crime neighborhoods: "
    f"{len(crime_neighborhood_set):,}"
)

print(
    f"CAD neighborhoods: "
    f"{len(call_neighborhood_set):,}"
)

print(
    f"Exact normalized overlap: "
    f"{len(shared_neighborhoods):,}"
)

print()

print("Crime-only neighborhood names:")
display(
    pd.DataFrame(
        {
            "crime_only": (
                crime_only_neighborhoods
            )
        }
    )
)

print("CAD-only neighborhood names:")
display(
    pd.DataFrame(
        {
            "calls_only": (
                calls_only_neighborhoods
            )
        }
    )
)

Crime neighborhoods: 59
CAD neighborhoods: 58
Exact normalized overlap: 58

Crime-only neighborhood names:


,crime_only
0,ooj


CAD-only neighborhood names:


,calls_only


In [ ]:
control_compatibility = pd.DataFrame(
    [
        {
            "metric": "Total offenses",
            "date": "Yes",
            "crime_type": "Yes",
            "subcategory": "Yes",
            "neighborhood": "Yes",
            "priority": "No",
        },
        {
            "metric": "Crime type totals / rates",
            "date": "Yes",
            "crime_type": "Yes",
            "subcategory": "Yes",
            "neighborhood": "Yes",
            "priority": "No",
        },
        {
            "metric": "Neighborhood crime ranking",
            "date": "Yes",
            "crime_type": "Yes",
            "subcategory": "Yes",
            "neighborhood": "Yes",
            "priority": "No",
        },
        {
            "metric": "Map coverage",
            "date": "Yes",
            "crime_type": "Yes",
            "subcategory": "Yes",
            "neighborhood": "Yes",
            "priority": "No",
        },
        {
            "metric": "Shootings",
            "date": "Yes",
            "crime_type": "Yes",
            "subcategory": "Yes",
            "neighborhood": "Yes",
            "priority": "No",
        },
        {
            "metric": "CAD response time",
            "date": "Yes",
            "crime_type": (
                "No validated offense/CAD link"
            ),
            "subcategory": (
                "No validated offense/CAD link"
            ),
            "neighborhood": (
                "Yes after geography validation"
            ),
            "priority": "Yes",
        },
    ]
)

display(control_compatibility)

,metric,date,crime_type,subcategory,neighborhood,priority
0,Total offenses,Yes,Yes,Yes,Yes,No
1,Crime type totals / rates,Yes,Yes,Yes,Yes,No
2,Neighborhood crime ranking,Yes,Yes,Yes,Yes,No
3,Map coverage,Yes,Yes,Yes,Yes,No
4,Shootings,Yes,Yes,Yes,Yes,No
5,CAD response time,Yes,No validated offense/CAD link,No validated offense/CAD link,Yes after geography validation,Yes


In [ ]:
canonical_type_ready = (
    len(unexpected_types) == 0
)

comparison_history_ready = (
    periods["previous_start"]
    >= EARLIEST_CRIME_DATE
)

population_ready = (
    CITYWIDE_POPULATION is not None
    and POPULATION_SOURCE_CONFIRMED
)

shooting_ready = (
    len(
        unresolved_shooting_values
    ) == 0
    and (
        MISSING_SHOOTING_MEANS_NO
        is not None
    )
    and SHOOTING_COUNT_UNIT
    in {
        "offense",
        "report",
    }
)

response_reproduction_ready = (
    existing_response_ids
    == reconstructed_qualified_ids
)


methodology_status = pd.DataFrame(
    {
        "methodology": [
            "Canonical crime bins",
            "Equal previous-period history",
            "Citywide population denominator",
            "Shooting definition",
            "Current response pipeline reproduced",
            "Dynamic map-coverage metric",
        ],
        "ready": [
            canonical_type_ready,
            comparison_history_ready,
            population_ready,
            shooting_ready,
            response_reproduction_ready,
            True,
        ],
        "note": [
            (
                "All included offenses should use "
                "exactly one of the three v1.1 bins."
            ),
            (
                "Previous period must exist for "
                "the selected current period."
            ),
            (
                "Requires documented population "
                "source before rates go live."
            ),
            (
                "Requires field-value semantics "
                "and offense-vs-report decision."
            ),
            (
                "Notebook event IDs should exactly "
                "match current production logic."
            ),
            (
                "Use filtered analytical population; "
                "do not exclude unmappable offenses "
                "from other metrics."
            ),
        ],
    }
)

display(methodology_status)


print()
print("CURRENT METHODOLOGY CANDIDATES")
print("=" * 60)

print(
    "Crime counting unit: unique offense_id"
)

print(
    "Previous comparison: immediately preceding "
    "equal-length period"
)

print(
    "Neighborhood crime ranking: unique offense volume"
)

print(
    "Response-time counting unit: unique CAD event"
)

print(
    "Candidate response priorities:",
    QUALIFIED_RESPONSE_PRIORITIES,
)

print(
    "Current candidate max response:",
    MAX_QUALIFIED_RESPONSE_MINUTES,
    "minutes",
)

print(
    "Candidate minimum neighborhood response sample:",
    MIN_NEIGHBORHOOD_RESPONSE_EVENTS,
)

print()

if not pd.isna(
    current_map_coverage[
        "unmappable_pct"
    ]
):
    print(
        "Current example unmappable share:",
        f"{current_map_coverage['unmappable_pct']:.2f}%",
    )

print()
print(
    "Report-before-offense timestamp anomalies from "
    "the prior QA do not affect ordinary offense-count "
    "metrics; they remain relevant if reporting-lag "
    "analysis is introduced later."
)

,methodology,ready,note
0,Canonical crime bins,True,All included offenses should use exactly one of the three v1.1 bins.
1,Equal previous-period history,True,Previous period must exist for the selected current period.
2,Citywide population denominator,False,Requires documented population source before rates go live.
3,Shooting definition,False,Requires field-value semantics and offense-vs-report decision.
4,Current response pipeline reproduced,True,Notebook event IDs should exactly match current production logic.
5,Dynamic map-coverage metric,True,Use filtered analytical population; do not exclude unmappable offenses from other metrics.



CURRENT METHODOLOGY CANDIDATES
Crime counting unit: unique offense_id
Previous comparison: immediately preceding equal-length period
Neighborhood crime ranking: unique offense volume
Response-time counting unit: unique CAD event
Candidate response priorities: [1, 2, 3]
Current candidate max response: 1440 minutes
Candidate minimum neighborhood response sample: 30

Current example unmappable share: 16.25%

Report-before-offense timestamp anomalies from the prior QA do not affect ordinary offense-count metrics; they remain relevant if reporting-lag analysis is introduced later.
